# SIH26168 — Dead Reckoning Training (Colab)

Trains the physics-informed bias-correction network on the IO-VNBD dataset.
This notebook is auto-generated from the local project's `src/` files
(`scripts/build_colab_notebook.py`) — the code cells below are byte-identical
to the repo, not a hand-copied version.

**Before running:** Runtime → Change runtime type → **T4 GPU** (or better).

**Steps:** get dataset → install deps → write project code → confirm real
column names → resume from the best existing checkpoint → train in a loop
until the PS's <10% drift target is hit (or this runtime ends) → evaluate →
export ONNX.


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("⚠️  No GPU — go to Runtime → Change runtime type → GPU before continuing.")


## 1. Get the dataset (IO-VNBD, ~1.6GB of CSVs over git-lfs)

In [ ]:
!apt-get -qq update && apt-get -qq install -y git-lfs
!git lfs install
!git clone --depth 1 https://github.com/onyekpeu/IO-VNBD.git data/IO-VNBD
%cd data/IO-VNBD
!git lfs pull --include="*.csv" --exclude="*.zip"
%cd /content


## 2. Mount Google Drive (recommended)

Keeps checkpoints/results if the Colab runtime disconnects. Skip this cell
if you'd rather not connect Drive — training still works, you'll just lose
checkpoints on disconnect.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CKPT_DIR = "/content/drive/MyDrive/SIH26168-ML/checkpoints"
RESULTS_DIR = "/content/drive/MyDrive/SIH26168-ML/results"
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Checkpoints will be saved to:", CKPT_DIR)


## 3. Install Python deps (torch is preinstalled on Colab; add the rest)

In [ ]:
!pip install -q onnx onnxruntime filterpy pyyaml


## 4. Project code

Written from the local repo's `src/` files — identical logic to what runs
on your machine, so results here are directly comparable.


In [ ]:
import os
os.makedirs("src/data", exist_ok=True)
os.makedirs("src/models", exist_ok=True)
os.makedirs("configs", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("results", exist_ok=True)
open("src/__init__.py", "w").close()
open("src/data/__init__.py", "w").close()
open("src/models/__init__.py", "w").close()


In [ ]:
%%writefile src/calibration.py
"""
Phone-to-vehicle frame calibration/alignment.

The PS explicitly calls out "in-vehicle phone alignment/calibration" as a
named component, and covers both dashboard-mounted and hand-held-in-holder
placement. A phone's IMU reports in the *phone's* body frame, which is
rotated arbitrarily relative to the car (mount angle, phone orientation,
whether it's in portrait/landscape). Before any dead-reckoning math means
anything, we need to rotate raw IMU samples into a consistent vehicle
frame: x = forward, y = left, z = up.

Two-stage estimate, standard for this problem:
  1. Leveling (roll/pitch) from gravity: while the vehicle is stationary
     (or from the low-pass/gravity component of accel while moving), the
     accelerometer measures ~[0,0,g] in the *level* frame. The tilt
     between that and the raw phone reading gives roll/pitch.
  2. Yaw (heading) misalignment: the phone's x-axis heading offset from
     the vehicle's forward direction isn't observable from gravity alone.
     We estimate it by comparing the direction of horizontal acceleration
     during a straightish accelerating/braking segment against the
     GPS-derived heading (or, at inference in the field with no GPS yet,
     we require a short calibration drive/segment before blackout, same
     as the PS's "instant seamless mode-switching" implies: leveling
     happens continuously, yaw-alignment happens once at trip start).

This module gives you the rotation only; it does not integrate anything
(see strapdown_ins.py for that).
"""
from __future__ import annotations

import numpy as np


def estimate_gravity_vector(accel: np.ndarray, method: str = "mean") -> np.ndarray:
    """Estimate the gravity direction from a (mostly) stationary accel window.

    Args:
        accel: (N, 3) raw accelerometer samples, m/s^2, phone frame.
        method: "mean" (assumes truly stationary) — swap for a low-pass
            filter if you calibrate during slow motion instead.

    Returns:
        (3,) unit gravity vector in the phone frame.
    """
    g = accel.mean(axis=0) if method == "mean" else accel[0]
    norm = np.linalg.norm(g)
    if norm < 1e-6:
        raise ValueError("Degenerate gravity estimate — accel window is all zero.")
    return g / norm


def leveling_rotation(gravity_phone: np.ndarray) -> np.ndarray:
    """Rotation matrix that maps the phone frame to a level frame
    (z-axis aligned with true vertical), leaving yaw unresolved.

    Uses the standard "align two vectors" construction (Rodrigues) mapping
    gravity_phone -> [0, 0, -1] (accelerometer reads -g when level & still,
    ENU-style: z up, gravity vector measured pointing up due to reaction force).
    """
    target = np.array([0.0, 0.0, 1.0])
    v = gravity_phone / np.linalg.norm(gravity_phone)

    axis = np.cross(v, target)
    s = np.linalg.norm(axis)
    c = np.dot(v, target)

    if s < 1e-8:
        # Already aligned (or exactly opposite) — no rotation needed / undefined axis.
        return np.eye(3) if c > 0 else np.diag([1.0, -1.0, -1.0])

    axis = axis / s
    K = np.array([
        [0, -axis[2], axis[1]],
        [axis[2], 0, -axis[0]],
        [-axis[1], axis[0], 0],
    ])
    R = np.eye(3) + K * s + (K @ K) * (1 - c)
    return R


def yaw_rotation(psi: float) -> np.ndarray:
    """Rotation about z by yaw angle psi (radians)."""
    c, s = np.cos(psi), np.sin(psi)
    return np.array([
        [c, -s, 0],
        [s, c, 0],
        [0, 0, 1],
    ])


def estimate_yaw_misalignment(level_accel_xy: np.ndarray, gt_heading_xy: np.ndarray) -> float:
    """Estimate constant yaw offset between the leveled phone frame and the
    vehicle forward direction, using a segment with known GT heading
    (e.g. GPS course-over-ground during a straight accelerating run).

    Args:
        level_accel_xy: (N, 2) horizontal accel in the *leveled* phone frame.
        gt_heading_xy: (N, 2) unit forward-direction vectors from GT (e.g.
            GPS bearing converted to a unit vector) for the same samples.

    Returns:
        psi: yaw offset in radians such that yaw_rotation(psi) applied to
        the leveled frame aligns it with the vehicle forward axis.
    """
    # Use the accel direction during the highest-magnitude (most confident)
    # samples only — near-zero accel gives an ill-defined direction.
    mag = np.linalg.norm(level_accel_xy, axis=1)
    keep = mag > np.percentile(mag, 75)
    a = level_accel_xy[keep]
    g = gt_heading_xy[keep]

    a_ang = np.arctan2(a[:, 1], a[:, 0])
    g_ang = np.arctan2(g[:, 1], g[:, 0])
    diff = np.arctan2(np.sin(g_ang - a_ang), np.cos(g_ang - a_ang))
    return float(np.median(diff))


def calibrate(accel: np.ndarray, gyro: np.ndarray, R_level: np.ndarray, psi_yaw: float = 0.0):
    """Apply the full phone->vehicle rotation to raw IMU streams.

    Args:
        accel, gyro: (N, 3) raw phone-frame samples.
        R_level: from leveling_rotation().
        psi_yaw: from estimate_yaw_misalignment(), 0.0 if not yet calibrated.

    Returns:
        accel_v, gyro_v: (N, 3) in the vehicle frame [forward, left, up].
    """
    R = yaw_rotation(psi_yaw) @ R_level
    accel_v = accel @ R.T
    gyro_v = gyro @ R.T
    return accel_v, gyro_v


In [ ]:
%%writefile src/models/strapdown_ins.py
"""
Differentiable 2D dead-reckoning integrator for ground vehicles.

Why 2D and not full 3D strapdown:
    A vehicle on a road is a non-holonomic system — it can't move sideways
    or vertically relative to itself. That constraint (explicitly called
    out in the PS) means we don't need a full 3D strapdown mechanization
    with quaternion attitude propagation; we only need:
        1. heading (yaw) angle, tracked by integrating calibrated yaw rate
        2. forward speed, estimated from calibrated forward acceleration
    and then dead-reckon x/y from heading + speed. This is the standard
    approach in vehicle/pedestrian inertial dead reckoning (RIDI, IONet,
    and the WhONet-style work the IO-VNBD dataset itself was built for).

This module is written to be differentiable (pure torch ops, no in-place
numpy) so it can sit *inside* the training loop: the network's residual
corrections feed into this integrator, and the loss is computed on the
integrated trajectory, not just on raw network output. That's what makes
this "physics-informed" rather than a black-box regressor.

The same function (without gradient tracking) is used again at inference
time on-device, which is why it takes plain tensors in/out and has no
framework-specific bells on it — this file is what gets carried into the
ONNX export path.
"""
from __future__ import annotations

import torch


def integrate_heading(yaw_rate: torch.Tensor, dt: float, theta0: torch.Tensor | float = 0.0) -> torch.Tensor:
    """Integrate yaw rate (rad/s) into heading (rad) over time.

    Args:
        yaw_rate: (batch, T) calibrated gyro-z, already in the vehicle frame.
        dt: sample period in seconds.
        theta0: initial heading, scalar or (batch,).

    Returns:
        theta: (batch, T) heading at each timestep (theta[:,0] is the first
        *post*-integration sample, i.e. theta0 + yaw_rate[:,0]*dt).
    """
    if not torch.is_tensor(theta0):
        theta0 = torch.full((yaw_rate.shape[0],), float(theta0), device=yaw_rate.device, dtype=yaw_rate.dtype)
    dtheta = yaw_rate * dt
    theta = theta0.unsqueeze(1) + torch.cumsum(dtheta, dim=1)
    return theta


def integrate_speed(forward_accel: torch.Tensor, dt: float, v0: torch.Tensor | float = 0.0,
                     clamp_min: float = 0.0) -> torch.Tensor:
    """Integrate forward (vehicle x-axis) acceleration into speed.

    clamp_min=0.0 enforces the non-holonomic / no-reverse assumption used
    by the PS's demo scenarios (forward driving only). Set to None to allow
    negative speed (reversing) if you extend this later.
    """
    if not torch.is_tensor(v0):
        v0 = torch.full((forward_accel.shape[0],), float(v0), device=forward_accel.device, dtype=forward_accel.dtype)
    dv = forward_accel * dt
    v = v0.unsqueeze(1) + torch.cumsum(dv, dim=1)
    if clamp_min is not None:
        v = torch.clamp(v, min=clamp_min)
    return v


def dead_reckon_position(speed: torch.Tensor, heading: torch.Tensor, dt: float,
                          p0: torch.Tensor | None = None) -> torch.Tensor:
    """Integrate speed+heading into a 2D nav-frame trajectory.

    Args:
        speed: (batch, T) forward speed, m/s.
        heading: (batch, T) heading, rad, same convention (0 = +x/east).
        dt: sample period, s.
        p0: (batch, 2) starting position, defaults to origin.

    Returns:
        positions: (batch, T, 2) [x, y] at each timestep.
    """
    batch = speed.shape[0]
    if p0 is None:
        p0 = torch.zeros(batch, 2, device=speed.device, dtype=speed.dtype)

    vx = speed * torch.cos(heading)
    vy = speed * torch.sin(heading)
    dx = vx * dt
    dy = vy * dt
    x = p0[:, 0:1] + torch.cumsum(dx, dim=1)
    y = p0[:, 1:2] + torch.cumsum(dy, dim=1)
    return torch.stack([x, y], dim=-1)


def drift_metric(pred_pos: torch.Tensor, gt_pos: torch.Tensor) -> torch.Tensor:
    """% positional drift at the end of each sequence, relative to distance travelled.

    This mirrors the PS benchmark directly: "positional drift < 10% of
    distance travelled during GNSS blackout".

    Args:
        pred_pos: (batch, T, 2)
        gt_pos:   (batch, T, 2)

    Returns:
        (batch,) drift percentage per sequence.
    """
    final_error = torch.linalg.norm(pred_pos[:, -1, :] - gt_pos[:, -1, :], dim=-1)
    step_dist = torch.linalg.norm(gt_pos[:, 1:, :] - gt_pos[:, :-1, :], dim=-1)
    distance_travelled = step_dist.sum(dim=1).clamp(min=1e-6)
    return 100.0 * final_error / distance_travelled


class DeadReckoner:
    """Convenience wrapper bundling calibrated-IMU -> trajectory in one call.

    Kept as a plain class (not nn.Module) since it holds no learnable
    parameters — corrections come from BiasCorrectionNet and are passed in.
    """

    def __init__(self, dt: float):
        self.dt = dt

    def integrate(self, forward_accel: torch.Tensor, yaw_rate: torch.Tensor,
                  v0: torch.Tensor | float = 0.0, theta0: torch.Tensor | float = 0.0,
                  p0: torch.Tensor | None = None):
        speed = integrate_speed(forward_accel, self.dt, v0=v0)
        heading = integrate_heading(yaw_rate, self.dt, theta0=theta0)
        pos = dead_reckon_position(speed, heading, self.dt, p0=p0)
        return speed, heading, pos


In [ ]:
%%writefile src/models/bias_correction_net.py
"""
BiasCorrectionNet — the learned part of the hybrid dead-reckoning pipeline.

Design (matches the "AI speed/vibration filter" line item in the PS):
    Raw MEMS IMU on a phone has time-varying bias and vibration-induced
    noise that a fixed physics model can't remove. Instead of trying to
    hand-tune that away, this network looks at a sliding window of
    calibrated IMU samples and predicts a *residual correction* to what
    the physics-only integrator (strapdown_ins.py) would have produced:

        delta_v      -- correction to forward-speed rate (m/s^2 equiv.)
        delta_theta  -- correction to yaw rate (rad/s)

    These residuals get added to the calibrated accel/gyro *before*
    integration in the training loop (see train.py), so the whole thing
    is trained end-to-end against real trajectory drift, not just against
    a hand-picked "bias" target that we don't actually have ground truth
    for.

Architecture: small 1D-CNN (local vibration/shape features) feeding a GRU
(temporal context across the window) feeding a linear head. This is the
same family used in published IMU-only odometry work (IONet, RIDI) and is
light enough to convert to ONNX -> ONNX Runtime Mobile / TFLite and run
at 10Hz on a phone with room to spare.

A "v2" variant (dilated residual CNN, 64/128/256 channels + deeper
256-hidden 3-layer GRU, ~1.9M params) was tried after v1 plateaued on
*train* drift across 6 warm-restart cycles (evidence of underfitting, not
overfitting — see results/train_history_6ch_baseline.json). Result:
71.35% test drift on a real Colab run, worse than v1's 62.76% — reverted
back to this file. See commit 8cd609e for the full v2 code/rationale if
revisiting; a next attempt would want to isolate *why* it underperformed
(e.g. residual/dilation structure vs. just more parameters) rather than
changing several things in one shot.
"""
from __future__ import annotations

import torch
import torch.nn as nn


class BiasCorrectionNet(nn.Module):
    def __init__(
        self,
        input_channels: int = 6,
        cnn_channels: list[int] | None = None,
        cnn_kernel_size: int = 5,
        gru_hidden: int = 128,
        gru_layers: int = 2,
        dropout: float = 0.2,
        output_dim: int = 2,
    ):
        super().__init__()
        cnn_channels = cnn_channels or [32, 64, 64]

        conv_layers = []
        in_ch = input_channels
        for out_ch in cnn_channels:
            conv_layers += [
                nn.Conv1d(in_ch, out_ch, kernel_size=cnn_kernel_size, padding=cnn_kernel_size // 2),
                nn.BatchNorm1d(out_ch),
                nn.ReLU(inplace=True),
            ]
            in_ch = out_ch
        self.cnn = nn.Sequential(*conv_layers)

        self.gru = nn.GRU(
            input_size=in_ch,
            hidden_size=gru_hidden,
            num_layers=gru_layers,
            batch_first=True,
            dropout=dropout if gru_layers > 1 else 0.0,
        )

        self.head = nn.Sequential(
            nn.Linear(gru_hidden, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(64, output_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (batch, T, C) calibrated IMU window — [ax, ay, az, gx, gy, gz]
               in the leveled vehicle frame (see calibration.py), plus
               engineered channels appended after (see windowing.py's
               engineer_features()) when C > 6. input_channels is a config
               value, not hardcoded, so this works either way.

        Returns:
            corrections: (batch, T, output_dim) per-timestep residuals
            [delta_v, delta_theta], same length as the input window so it
            can be added directly to the physics integrator's inputs.
        """
        # Conv1d wants (batch, C, T)
        h = self.cnn(x.transpose(1, 2)).transpose(1, 2)  # -> (batch, T, C')
        h, _ = self.gru(h)                                 # -> (batch, T, gru_hidden)
        out = self.head(h)                                 # -> (batch, T, output_dim)
        return out


In [ ]:
%%writefile src/data/io_vnbd_loader.py
"""
IO-VNBD raw CSV -> structured numpy arrays.

Confirmed against the real headers (see inspect output, 2026-09-03): this
dataset has two genuinely different file types living in the same folders,
distinguished by filename prefix:

  - `V-*.csv` — vehicle CAN-bus data. Has "Indicated Longitudinal/Lateral
    Acceleration" (2-axis, in g) and "Yaw Rate" (1-axis), but no full
    3-axis accelerometer/gyroscope. Not usable for this pipeline, which
    needs 6-axis IMU — these files are expected to fail to resolve and get
    skipped (see windowing.py's `file_prefix` filter, which excludes them
    up front instead of relying on the per-file skip).

  - `S-*.csv` — smartphone recordings. Has real 3-axis accelerometer +
    gyroscope + GPS, which is exactly the "phone in the car" scenario this
    PS is about. This is the file type we actually train on.

Two more real quirks confirmed from the headers:
  - Gyro axes are labeled inconsistently across sub-folders: some files
    use `GYROSCOPE X/Y/Z (rad/s)`, others use `GYROSCOPE Yaw/Pitch/Roll
    (rad/s)`. We treat Yaw=Z (yaw rate is literally what strapdown_ins.py
    uses as heading rate), Pitch=Y, Roll=X — the standard vehicle-axis
    convention, and a reasonable read of a differently-labeled but
    equivalent quantity.
  - Units aren't SI everywhere: "GPS SPEED (Kmh)" is km/h, not m/s;
    "TIME SINCE START (ms)" is milliseconds, not seconds. Both get
    converted to SI right after column resolution, so everything
    downstream (windowing.py, strapdown_ins.py) can assume SI units.

Column resolution itself:
  1. tries configs/default.yaml's `column_map` if it's been filled in, and
  2. falls back to fuzzy substring matching on normalized column names.

Everything downstream (windowing.py, calibration.py, strapdown_ins.py)
consumes the plain dict this returns, not raw DataFrames, so a schema
change here doesn't ripple through the rest of the code.
"""
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

# substring -> canonical field, checked against normalized headers (spaces
# become underscores, not stripped — "ACCELEROMETER X (m/s²)" normalizes to
# "accelerometer_x_(m/s²)", which is what these patterns are written against)
_FUZZY_PATTERNS: dict[str, list[str]] = {
    "time": ["time_since_start", "timestamp", "time", "t(s)", "t_s"],
    "accel_x": ["accelerometer_x", "acc_x", "accel_x", "accx"],
    "accel_y": ["accelerometer_y", "acc_y", "accel_y", "accy"],
    "accel_z": ["accelerometer_z", "acc_z", "accel_z", "accz"],
    # Yaw/Pitch/Roll variant seen in some sub-folders (see module docstring)
    # checked before the bare single-letter patterns so it isn't shadowed.
    "gyro_x": ["gyroscope_roll", "gyroscope_x", "gyro_x", "gyrox"],
    "gyro_y": ["gyroscope_pitch", "gyroscope_y", "gyro_y", "gyroy"],
    "gyro_z": ["gyroscope_yaw", "gyroscope_z", "gyro_z", "gyroz", "yaw_rate"],
    "speed_gt": ["gps_speed", "wheel_speed", "wheelspeed", "obd_speed", "speed", "velocity"],
    "lat": ["latitude", "lat"],
    "lon": ["longitude", "lon", "lng"],
    "heading_gt": ["gps_orientation", "heading", "course", "bearing"],
}


@dataclass
class ImuSequence:
    path: Path
    time: np.ndarray            # (N,) seconds
    accel: np.ndarray           # (N,3) [ax, ay, az] m/s^2, raw phone/vehicle frame
    gyro: np.ndarray            # (N,3) [gx, gy, gz] rad/s
    speed_gt: np.ndarray | None  # (N,) m/s, ground truth (wheel-speed / GPS-derived)
    lat: np.ndarray | None
    lon: np.ndarray | None
    heading_gt: np.ndarray | None

    @property
    def n(self) -> int:
        return len(self.time)


def _match_column(columns: list[str], patterns: list[str]) -> str | None:
    # underscore (not strip) so "ACCELEROMETER X (...)" -> "accelerometer_x_(...)"
    # keeps the word boundary the patterns above are written against.
    lower = {c.lower().strip().replace(" ", "_").replace("-", "_"): c for c in columns}
    for pat in patterns:
        for lc, orig in lower.items():
            if pat in lc:
                return orig
    return None


def resolve_columns(df: pd.DataFrame, column_map: dict[str, str | None] | None = None) -> dict[str, str | None]:
    """Resolve canonical field -> actual column name, preferring an explicit
    config mapping and falling back to fuzzy matching."""
    resolved: dict[str, str | None] = {}
    columns = list(df.columns)
    for field, patterns in _FUZZY_PATTERNS.items():
        explicit = (column_map or {}).get(field)
        if explicit and explicit in columns:
            resolved[field] = explicit
        else:
            resolved[field] = _match_column(columns, patterns)
    return resolved


def _kmh_header_unit_factor(speed: np.ndarray, lat: np.ndarray | None, lon: np.ndarray | None,
                            time_s: np.ndarray) -> float:
    """Multiplier that turns a column headed "GPS SPEED (Kmh)" into m/s —
    decided from the file's own data, not from the header.

    Real bug found this session, after months of it sitting in plain sight:
    IO-VNBD's phone files label this column "Kmh", but the values are
    metres per second. Checked on every file where the comparison is
    possible (108 of 144): integrating the column as m/s lands within 6% of
    the GPS lat/lon path length for the median file (p10-p90: 0.94-1.06),
    93% of files sit within 15% of exactly 1.00, and only 2% look like
    genuine km/h (ratio near 3.6). The old rule (divide by 3.6 whenever the
    header says Kmh) therefore made every IO-VNBD speed label 3.6x too
    small: the physics integrator started each window at a third of the
    true speed, speed_loss pushed the network toward a deflated target
    while drift_loss pushed it toward the real (undeflated) position track,
    and docs/understanding.md's "IO-VNBD is urban, low-speed" framing was
    partly an artifact of speeds that were 3.6x too low (its real median
    speed is ~16 m/s, not ~4).

    Compares the path length of the GPS track with the speed column's own
    integral treated as m/s: a ratio near 1 means m/s, near 3.6 means
    genuine km/h. Falls back to m/s (what 93% of the evidence supports,
    not the header's claim) when the file has too little usable GPS to
    decide."""
    default = 1.0
    if lat is None or lon is None or time_s is None or len(speed) < 50:
        return default
    ok = np.isfinite(lat) & np.isfinite(lon) & np.isfinite(speed)
    if ok.sum() < 50:
        return default
    duration = float(time_s[-1] - time_s[0])
    if not duration > 30.0:
        return default
    xy = latlon_to_local_xy(lat[ok], lon[ok])
    path_m = float(np.linalg.norm(np.diff(xy, axis=0), axis=1).sum())
    integral_as_ms = float(np.sum(speed[ok]) * duration / len(speed))
    if path_m < 50.0 or integral_as_ms < 50.0:
        return default
    ratio = path_m / integral_as_ms
    # nearest of the two candidate units, compared in log space
    return 1.0 if abs(np.log(ratio)) <= abs(np.log(ratio / 3.6)) else 1.0 / 3.6


def _fill_held_samples(time_s: np.ndarray, arrays: list[np.ndarray],
                       circular_deg: bool = False) -> list[np.ndarray]:
    """Turn a held-then-jump (stair-step) GPS signal into a continuous one by
    linearly interpolating between the rows where a new value actually
    arrived.

    Real problem found this session: IO-VNBD's phone files log GPS at the
    10Hz sensor rate, but the GPS itself updates far less often, and the
    last fix is simply repeated on every row in between. Measured over 136
    files, in 96% of them more than a fifth of the gaps between new
    positions exceed 3 s, and the typical worst-case gap is about 9 s. The
    position label was therefore a staircase: flat for seconds, then a jump.
    Two consequences, both fixed by this function:
      - a 5 s window's end-point "truth" was often seconds stale, so the
        drift metric was scored against a position the car had already left
        (windows that caught a jump showed ~1.8x the distance the speed
        label said; windows that did not were flat and got thrown out by
        build_windows as "parked"); and
      - build_windows' note that ~47% of windows are "essentially
        stationary ... bimodal, nothing in between" was describing this
        artifact (the speed label shows those windows moving at ~7 m/s),
        not idling cars.

    Rows where a value is NaN (real GPS blackout stretches in own recordings)
    are left NaN: interpolation only happens inside each contiguous run of
    valid rows, never across a gap, so build_windows' skip-windows-with-NaN
    logic still sees real blackouts as blackouts.

    arrays share one change mask (a row counts as "new" if ANY of them
    changed), which is what you want for lat+lon that arrive together. A
    column that changes on every row comes back unchanged. circular_deg
    interpolates compass degrees through unit vectors so 358 -> 3 does not
    swing through 180. Interpolating between arrival times lags the true
    motion by the phone's GPS latency (a fraction of a second); that is a
    far smaller error than seconds of staleness."""
    arrays = [np.asarray(a, dtype=np.float64) for a in arrays]
    outs = [a.copy() for a in arrays]
    finite = np.all([np.isfinite(a) for a in arrays], axis=0)
    idx = np.flatnonzero(finite)
    if len(idx) < 3:
        return outs
    breaks = np.flatnonzero(np.diff(idx) > 1)
    starts = np.r_[idx[0], idx[breaks + 1]]
    ends = np.r_[idx[breaks], idx[-1]]
    for s, e in zip(starts, ends):
        if e - s < 2:
            continue
        seg_t = time_s[s:e + 1]
        changed = np.zeros(e - s + 1, dtype=bool)
        changed[0] = True
        for a in arrays:
            changed[1:] |= a[s + 1:e + 1] != a[s:e]
        if changed.all() or changed.sum() < 2:
            continue
        t_c = seg_t[changed]
        keep = np.r_[True, np.diff(t_c) > 0]  # np.interp needs strictly increasing x
        t_c = t_c[keep]
        for out, a in zip(outs, arrays):
            vals = a[s:e + 1][changed][keep]
            if circular_deg:
                r = np.radians(vals)
                c = np.interp(seg_t, t_c, np.cos(r))
                sn = np.interp(seg_t, t_c, np.sin(r))
                out[s:e + 1] = np.degrees(np.arctan2(sn, c)) % 360.0
            else:
                out[s:e + 1] = np.interp(seg_t, t_c, vals)
    return outs


def load_sequence(path: Path, column_map: dict[str, str | None] | None = None) -> ImuSequence:
    # IO-VNBD's CSVs aren't consistently UTF-8 (some contain stray bytes from
    # degree/superscript symbols in free-text fields) — latin-1 never raises
    # a decode error since it maps every byte 0-255, and the columns we
    # actually use here are numeric, so mis-decoded text elsewhere is harmless.
    try:
        df = pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding="latin-1")
    cols = resolve_columns(df, column_map)

    missing_required = [f for f in ("accel_x", "accel_y", "accel_z", "gyro_x", "gyro_y", "gyro_z") if cols[f] is None]
    if missing_required:
        raise ValueError(
            f"{path}: could not resolve required IMU columns {missing_required}. "
            f"Available columns: {list(df.columns)}. "
            f"Fill in configs/default.yaml's data.column_map with the real names "
            f"(run src/data/inspect_dataset.py to see them)."
        )

    def col(field):
        c = cols[field]
        return df[c].to_numpy(dtype=np.float64) if c else None

    time = col("time")
    if time is None:
        # no explicit timestamp column — assume uniform sampling, filled in by caller via sample_rate
        time = np.arange(len(df), dtype=np.float64)
    elif cols["time"] and "(ms)" in cols["time"].lower():
        time = time / 1000.0  # -> seconds

    accel = np.stack([col("accel_x"), col("accel_y"), col("accel_z")], axis=1)
    gyro = np.stack([col("gyro_x"), col("gyro_y"), col("gyro_z")], axis=1)

    lat = col("lat")
    lon = col("lon")
    speed_gt = col("speed_gt")
    heading_gt = col("heading_gt")
    if speed_gt is not None and cols["speed_gt"] and "kmh" in cols["speed_gt"].lower().replace("/", ""):
        speed_gt = speed_gt * _kmh_header_unit_factor(speed_gt, lat, lon, time)

    # Held GPS samples -> continuous track (see _fill_held_samples). Position
    # fixes arrive together, so lat/lon share one change mask; speed and
    # heading are detected independently (a column that really does change
    # every row is left untouched).
    if lat is not None and lon is not None:
        lat, lon = _fill_held_samples(time, [lat, lon])
    if speed_gt is not None:
        (speed_gt,) = _fill_held_samples(time, [speed_gt])
    if heading_gt is not None:
        (heading_gt,) = _fill_held_samples(time, [heading_gt], circular_deg=True)

    return ImuSequence(
        path=path,
        time=time,
        accel=accel,
        gyro=gyro,
        speed_gt=speed_gt,
        lat=lat,
        lon=lon,
        heading_gt=heading_gt,
    )


def latlon_to_local_xy(lat: np.ndarray, lon: np.ndarray) -> np.ndarray:
    """Equirectangular projection to local metres, referenced to the first
    point in the sequence. Good enough over a single trip's extent (a few
    km at most) — not meant for anything requiring true geodesy."""
    lat0, lon0 = lat[0], lon[0]
    R = 6371000.0
    lat0_rad = np.radians(lat0)
    x = np.radians(lon - lon0) * R * np.cos(lat0_rad)
    y = np.radians(lat - lat0) * R
    return np.stack([x, y], axis=1)


def compass_deg_to_xy_unit(bearing_deg: np.ndarray) -> np.ndarray:
    """Convert a compass bearing (0°=North, 90°=East, clockwise — GPS
    course-over-ground convention) into a unit vector in the [x=east,
    y=north] frame latlon_to_local_xy uses, so heading_gt is directly
    comparable to position-derived directions without a separate
    degrees<->math-angle conversion step at every call site."""
    rad = np.radians(bearing_deg)
    return np.stack([np.sin(rad), np.cos(rad)], axis=-1)


def derive_speed_from_gps(lat: np.ndarray, lon: np.ndarray, time: np.ndarray) -> np.ndarray:
    """Fallback ground-truth speed from consecutive GPS fixes, if the file
    has no direct wheel-speed / speed column."""
    xy = latlon_to_local_xy(lat, lon)
    d = np.linalg.norm(np.diff(xy, axis=0), axis=1)
    dt = np.diff(time)
    dt[dt <= 0] = np.nan
    v = d / dt
    v = np.concatenate([[v[0]], v])
    return np.nan_to_num(v, nan=0.0)


def recover_yaw_axis(seq: ImuSequence, horizon_s: float = 10.0, min_r: float = 0.6,
                     min_moving_windows: int = 400, min_speed_mps: float = 3.0) -> ImuSequence:
    """Rebuild a usable vehicle-yaw rate from IO-VNBD's phone gyro.

    Real input bug found by the label audit (src/audit_labels.py): the column
    the loader reads as yaw rate ("GYROSCOPE Yaw (rad/s)") has ~zero
    correlation with how the GPS heading actually changes (median -0.01 over
    10 s horizons across 29 files), while the accelerometer says the phone
    lies flat. The phone's gyro axes are evidently not in the accelerometer's
    frame in these logs, and the vertical rotation is smeared across all
    three gyro channels in a per-file way. Gravity-based leveling therefore
    cannot recover it, and the model has been trained on a garbage yaw signal
    for most IO-VNBD files: that, not "urban driving is hard", is a large
    part of the 40-60% urban drift.

    Fix: per file, least-squares fit  heading change over `horizon_s`
    ~ integrated-gyro change over the same span @ w  (moving > min_speed_mps),
    using the GPS course as the independent reference. The fitted 3-vector w
    maps the raw gyro onto the true vertical axis:
        gz_new = gyro @ w          (true yaw rate, fitted scale included)
        gx_new, gy_new = components of gyro along the two unit vectors
                         orthogonal to w (remaining roll/pitch rates)
    Files where the fit is poor (correlation R < min_r) have no recoverable
    yaw axis and raise ValueError, so callers' existing try/except skips them
    (and reports it) instead of training on / scoring against noise.

    This is a per-file 3-parameter calibration against GPS, the same role the
    app's live yaw alignment plays at the start of a trip (CalibrationManager
    uses GPS heading while moving). It is applied identically to train, val
    and test files. The deployed app reads a real Android gyro whose frame
    matches its accelerometer, so this is a dataset-specific repair, not
    something the app needs.
    """
    if seq.heading_gt is None or seq.speed_gt is None:
        raise ValueError(f"{seq.path}: no heading_gt/speed_gt to recover the yaw axis from")
    dt = float(np.median(np.diff(seq.time)))
    H = max(2, int(round(horizon_s / dt)))
    if seq.n < 3 * H:
        raise ValueError(f"{seq.path}: too short ({seq.n * dt:.0f}s) to recover the yaw axis")
    unit = compass_deg_to_xy_unit(seq.heading_gt)
    heading = np.unwrap(np.arctan2(unit[:, 1], unit[:, 0]))
    dh = heading[H:] - heading[:-H]
    cum = np.cumsum(seq.gyro.astype(np.float64), axis=0) * dt
    dG = cum[H:] - cum[:-H]
    mean_speed = np.convolve(np.asarray(seq.speed_gt, dtype=np.float64), np.ones(H) / H, mode="valid")[: len(dh)]
    ok = (mean_speed > min_speed_mps) & np.isfinite(dh) & np.isfinite(dG).all(axis=1)
    if ok.sum() < min_moving_windows:
        raise ValueError(f"{seq.path}: only {int(ok.sum())} moving {horizon_s:.0f}s spans, not enough to recover the yaw axis")
    w, *_ = np.linalg.lstsq(dG[ok], dh[ok], rcond=None)
    r = float(np.corrcoef(dG[ok] @ w, dh[ok])[0, 1])
    if not np.isfinite(r) or r < min_r:
        raise ValueError(f"{seq.path}: gyro yaw axis not recoverable (fit R={r:.2f} < {min_r})")
    wn = float(np.linalg.norm(w))
    z = w / wn
    a = np.array([1.0, 0.0, 0.0]) if abs(z[0]) < 0.9 else np.array([0.0, 1.0, 0.0])
    x = a - (a @ z) * z
    x /= np.linalg.norm(x)
    y = np.cross(z, x)
    g = seq.gyro.astype(np.float64)
    seq.gyro[:, 0] = (g @ x).astype(seq.gyro.dtype)
    seq.gyro[:, 1] = (g @ y).astype(seq.gyro.dtype)
    seq.gyro[:, 2] = (g @ w).astype(seq.gyro.dtype)
    return seq


In [ ]:
%%writefile src/data/windowing.py
"""
Turns raw IO-VNBD sequences into fixed-length training windows.

Split is done at the *sequence* (trip/file) level, not the window level —
windows from the same drive are highly correlated, so splitting after
windowing would leak train data into val/test and give an optimistic drift
number that won't hold up on demo day. That would be exactly the kind of
mistake worth catching before judges do.

Each window carries everything the training loop needs:
  - calibrated accel/gyro samples (network input)
  - forward-accel / yaw-rate the physics integrator would use on its own
    (network input's physical counterpart, so the residual correction has
    something to be a residual *of*)
  - GT speed at each step (supervises the per-step loss)
  - GT position relative to the window's start (supervises the drift loss)
  - v0 / theta0: the vehicle's true starting speed and heading for this
    window. A window is a random 5s slice of a drive — the vehicle is
    essentially never stopped and facing due-east right at the slice
    boundary, so the integrator MUST start from the real initial state,
    not from zero. (Confirmed the hard way: training without this
    plateaued around 85% drift no matter how long it ran — the model was
    being asked to predict trajectories from a starting condition that
    was wrong for nearly every window.)
"""
from __future__ import annotations

import glob
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset

from .io_vnbd_loader import (
    ImuSequence,
    compass_deg_to_xy_unit,
    derive_speed_from_gps,
    latlon_to_local_xy,
    load_sequence,
    recover_yaw_axis,
)
from ..calibration import calibrate, estimate_gravity_vector, estimate_yaw_misalignment, leveling_rotation


@dataclass
class Window:
    accel: np.ndarray       # (T, 3) calibrated, vehicle frame
    gyro: np.ndarray        # (T, 3) calibrated, vehicle frame
    speed_gt: np.ndarray    # (T,)
    pos_gt: np.ndarray      # (T, 2) relative to window start
    v0: float                # true starting speed, m/s
    theta0: float            # true starting heading, rad (world frame, matches pos_gt's axes)
    lat0: float               # true starting lat/lon — lets a predicted local-xy trajectory be
    lon0: float               # converted back to real coordinates for map-matching against OSM
    dt: float
    # Where this window came from, so chain_eval.py can stitch back-to-back
    # windows of one recording into a continuous blackout. Defaults keep
    # every other Window(...) caller working unchanged.
    src: str = ""            # source recording (str of seq.path)
    start: int = -1          # sample index of the window's first sample in that recording


def resample_uniform(seq: ImuSequence, target_hz: float) -> ImuSequence:
    """Resample an (possibly irregular-rate) sequence to a fixed dt via
    linear interpolation. IO-VNBD's phone stream is documented at 10Hz;
    the vehicle stream may differ, so this makes both trainable with the
    same window logic."""
    t0, t1 = seq.time[0], seq.time[-1]
    n = int((t1 - t0) * target_hz)
    if n < 2:
        return seq
    t_new = np.linspace(t0, t1, n)

    def interp(arr):
        if arr is None:
            return None
        if arr.ndim == 1:
            return np.interp(t_new, seq.time, arr)
        return np.stack([np.interp(t_new, seq.time, arr[:, i]) for i in range(arr.shape[1])], axis=1)

    def interp_heading_deg(deg):
        # Plain np.interp on raw degrees breaks at the 0/360 wraparound
        # (e.g. 358° -> 3° would linearly blend through 180°, the opposite
        # direction). Interpolate the compass bearing as a unit vector
        # instead — no wraparound to break — then convert back to degrees.
        if deg is None:
            return None
        rad = np.radians(deg)
        c = np.interp(t_new, seq.time, np.cos(rad))
        s = np.interp(t_new, seq.time, np.sin(rad))
        return np.degrees(np.arctan2(s, c)) % 360

    return ImuSequence(
        path=seq.path,
        time=t_new,
        accel=interp(seq.accel),
        gyro=interp(seq.gyro),
        speed_gt=interp(seq.speed_gt),
        lat=interp(seq.lat),
        lon=interp(seq.lon),
        heading_gt=interp_heading_deg(seq.heading_gt),
    )


def calibrate_sequence(seq: ImuSequence, stationary_samples: int = 20,
                        min_confident_samples: int = 3000, min_speed_mps: float = 2.0) -> ImuSequence:
    """Leveling (roll/pitch) + yaw calibration.

    First attempt at yaw calibration used the GPS track's own raw
    frame-to-frame position bearing as ground truth, and that made things
    worse — a single 0.1s GPS position delta at 10Hz is the same order of
    magnitude as consumer GPS position noise, so the estimates came back
    wildly inconsistent (-123° to +99° across similar files). This version
    uses the phone's own GPS-orientation field instead (course-over-ground,
    computed by the GPS chip itself, not our own differencing — confirmed
    far smoother: holds steady ~90% of samples between real fixes, updates
    sensibly when it does change) and filters "is it actually moving" by
    the GPS speed field directly rather than a noisy position-diff — both
    changes replace a self-inflicted noise source with the dataset's own,
    presumably better-filtered signal.
    """
    n = min(stationary_samples, len(seq.accel))
    gravity = estimate_gravity_vector(seq.accel[:n])
    R_level = leveling_rotation(gravity)

    psi_yaw = 0.0
    if seq.heading_gt is not None and seq.speed_gt is not None and len(seq.heading_gt) > min_confident_samples:
        confident = seq.speed_gt > min_speed_mps
        if confident.sum() >= min_confident_samples:
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore", category=RuntimeWarning)  # see note below
                level_accel = seq.accel @ R_level.T
            heading_xy = compass_deg_to_xy_unit(seq.heading_gt)
            psi_yaw = estimate_yaw_misalignment(level_accel[confident, :2], heading_xy[confident])

    # macOS's Accelerate BLAS backend emits spurious "divide by zero" /
    # "invalid value" RuntimeWarnings on these small (Nx3 @ 3x3) matmuls —
    # a known false-positive (verified: output stays fully finite; the FPE
    # flag it's reacting to comes from Accelerate's internal computation,
    # not the actual result). Doesn't happen on Linux/OpenBLAS (e.g. Colab).
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=RuntimeWarning)
        accel_v, gyro_v = calibrate(seq.accel, seq.gyro, R_level, psi_yaw)
    assert np.isfinite(accel_v).all() and np.isfinite(gyro_v).all(), (
        f"{seq.path}: calibration produced non-finite values for real — "
        f"this one isn't the benign Accelerate warning, don't ignore it."
    )
    seq.accel[:] = accel_v
    seq.gyro[:] = gyro_v
    return seq


def build_windows(seq: ImuSequence, window_size: int, stride: int, dt: float,
                   min_distance_m: float = 2.0) -> list[Window]:
    """min_distance_m excludes windows the vehicle was essentially parked/idle
    for. Real driving logs have a lot of this (confirmed: ~47% of windows in
    this dataset travel under 10cm in a 5s window — engine idling, traffic
    stops, before/after the actual drive). This isn't a minor edge case to
    smooth over: dividing a position error by near-zero distance travelled
    (drift_metric's whole definition, matching the PS's own "% of distance
    travelled" wording) blows up to enormous, meaningless percentages for
    those windows — confirmed empirically: one batch's *mean* train drift
    came out at 134,735% before this filter, entirely from a handful of such
    windows dominating the average, not from the model being that wrong.
    It also isn't the scenario the PS benchmarks anyway — blackout tracking
    while the vehicle is moving, not parked. 2m over a 5s window is a clean
    cut confirmed against the real distribution (bimodal: windows are either
    ~0m or already tens of metres, nothing in between)."""
    n = seq.n
    if seq.lat is not None and seq.lon is not None:
        xy = latlon_to_local_xy(seq.lat, seq.lon)
    else:
        xy = None

    if seq.speed_gt is not None:
        speed_gt = seq.speed_gt
    elif xy is not None:
        speed_gt = derive_speed_from_gps(seq.lat, seq.lon, seq.time)
    else:
        speed_gt = None

    windows = []
    for start in range(0, n - window_size, stride):
        end = start + window_size
        w_accel = seq.accel[start:end]
        w_gyro = seq.gyro[start:end]
        w_speed = speed_gt[start:end] if speed_gt is not None else None
        w_pos = (xy[start:end] - xy[start]) if xy is not None else None

        if w_speed is None or w_pos is None:
            continue  # can't supervise this window without GT — skip

        if not np.isfinite(w_speed).all() or not np.isfinite(w_pos).all():
            # Window overlaps a real GPS blackout gap (blank lat/lon/speed
            # fields -> NaN, see io_vnbd_loader.load_sequence) — no ground
            # truth to supervise it. Found the hard way: NaN comparisons
            # are always False in Python/numpy, so the min_distance_m check
            # right below this (`step_dist.sum() < min_distance_m`) does
            # NOT catch a NaN window — it silently let NaN speed_gt/pos_gt
            # through into training, which corrupted the whole model's
            # weights to NaN within the first batch that contained one.
            # Never triggered on IO-VNBD (its files have no mid-file GPS
            # dropouts), only surfaced once a real phone recording with a
            # genuine blackout stretch (data/own_recordings/, expected and
            # useful per that folder's README) got mixed into training.
            continue

        step_dist = np.linalg.norm(np.diff(w_pos, axis=0), axis=1)
        if step_dist.sum() < min_distance_m:
            continue  # essentially stationary — see docstring

        v0 = float(w_speed[0])
        # true starting heading: prefer the GPS-orientation field (GPS
        # chip's own course-over-ground, confirmed smooth — see
        # calibrate_sequence) at the window's start sample when the
        # vehicle's actually moving there. Falls back to bearing from the
        # position track's own first ~1s (short lookahead approximates
        # instantaneous direction of travel; the full window's endpoint
        # would blend in whatever turning happens later in the window,
        # which isn't "starting" heading) if heading_gt isn't available or
        # the window happens to start from a near-stop.
        if seq.heading_gt is not None and w_speed[0] > 1.0:
            unit = compass_deg_to_xy_unit(seq.heading_gt[start])
            theta0 = float(np.arctan2(unit[1], unit[0]))
        else:
            lookahead = min(10, window_size - 1)
            d = w_pos[lookahead] - w_pos[0]
            if np.linalg.norm(d) < 0.5:  # too little motion in that short a span — widen it
                d = w_pos[-1] - w_pos[0]
            theta0 = float(np.arctan2(d[1], d[0]))

        lat0 = float(seq.lat[start]) if seq.lat is not None else float("nan")
        lon0 = float(seq.lon[start]) if seq.lon is not None else float("nan")

        windows.append(Window(accel=w_accel, gyro=w_gyro, speed_gt=w_speed, pos_gt=w_pos,
                               v0=v0, theta0=theta0, lat0=lat0, lon0=lon0, dt=dt,
                               src=str(seq.path), start=start))
    return windows


def engineer_features(accel: np.ndarray, gyro: np.ndarray, dt: float, smooth_win: int = 5) -> np.ndarray:
    """Extra per-timestep channels appended after the raw 6 calibrated
    accel/gyro axes.

    IS now wired in — via IOVNBDWindowDataset's extra_features=True /
    src.train's --extra_features flag — after a proper, conclusive retry
    (2026-09-05) fixed both gaps the first inconclusive attempt had: it now
    gets real per-channel z-score normalization (computed from train-split
    stats only, see compute_channel_norm_stats — these 6 extra channels are
    on very different scales from each other and from the raw 6, unlike
    train.py's reasonably-scaled raw accel/gyro that trained fine into
    BatchNorm1d unnormalized) and a fair, generous early_stop_patience=15
    for the first cycle (configs/engineered_features.yaml) instead of the
    original attempt's default 8, which had cut it off at epoch 10 — nowhere
    near the 57 epochs the 6-channel baseline took to reach its own best.

    Conclusive result this time, combined with comma2k19 mixed into
    training (checkpoints/best_engineered.pt, results/eval_report saved
    under that run): **worse than the 6-channel baseline on the metric that
    matters most** — 62.75% combined / 64.28% IO-VNBD-only test drift, vs.
    60.92%/62.33% without the extra channels. The one genuine win: comma2k19
    -only test drift improved further to 14.48% mean / **5.42% median**,
    71.45% pass rate (vs. 16.49%/8.94%/54.5% with just the raw 6 channels)
    — the extra engineered channels (magnitude, jerk, local roughness) do
    help the model on already-easy, clean highway driving, but make the
    harder IO-VNBD urban/stop-and-go case worse, not better. **Reverted to
    the 6-channel baseline as the production checkpoint** (`checkpoints/
    best.pt`) given this; `checkpoints/best_engineered.pt` and its eval
    reports are kept as a reference, same treatment as best_6ch_baseline.pt.

    Four training cycles of warm-restarting the 6-raw-channel model against
    the same data all plateaued in the exact same ~68-70% val-drift band
    (see results/train_history_6ch_baseline.json), and train drift
    plateaued right along with it — evidence of underfitting from limited
    input signal (motivating this), not of overfitting or a bad LR.

    Appended *after* the 6 raw channels (never inserted before) so
    `train.py`'s `imu[..., 0]` (forward accel) and `imu[..., 5]` (yaw rate)
    indexing into the physics baseline stays correct, if re-enabled.
    """
    ax, ay, az = accel[:, 0], accel[:, 1], accel[:, 2]
    gx, gy, gz = gyro[:, 0], gyro[:, 1], gyro[:, 2]

    accel_mag = np.sqrt(ax**2 + ay**2 + az**2)
    accel_horiz_mag = np.sqrt(ax**2 + ay**2)
    gyro_mag = np.sqrt(gx**2 + gy**2 + gz**2)

    # Sharp brake/throttle transitions look different from steady
    # acceleration only in their rate of change — a single raw sample can't
    # tell them apart.
    jerk_x = np.gradient(ax, dt)

    # Local smoothing/roughness of the two channels the physics baseline
    # actually integrates: a short moving average (separates sustained
    # motion from a single noisy sample) and local std (vibration
    # intensity/roughness) over a ~0.5s window.
    kernel = np.ones(smooth_win) / smooth_win
    ax_smooth = np.convolve(ax, kernel, mode="same")
    ax_local_std = np.sqrt(np.clip(np.convolve(ax**2, kernel, mode="same") - ax_smooth**2, 0, None))
    gz_smooth = np.convolve(gz, kernel, mode="same")
    gz_local_std = np.sqrt(np.clip(np.convolve(gz**2, kernel, mode="same") - gz_smooth**2, 0, None))

    return np.stack(
        [accel_mag, accel_horiz_mag, gyro_mag, jerk_x, ax_local_std, gz_local_std], axis=1
    ).astype(np.float32)


class IOVNBDWindowDataset(Dataset):
    def __init__(self, windows: list[Window], extra_features: bool = False,
                 norm_mean: np.ndarray | None = None, norm_std: np.ndarray | None = None):
        """extra_features=True appends engineer_features()'s 6 extra channels
        (12 total). norm_mean/norm_std (per-channel, computed once from the
        *train* split only — see compute_channel_norm_stats — and reused
        across val/test/comma2k19_test_only) z-score the raw+engineered
        channels before they hit the network. Both were flagged as missing
        in the first, inconclusive attempt at this (see engineer_features()'s
        docstring) — the 6 raw channels are all similar-scale accel/gyro
        already, so they trained fine unnormalized, but jerk/local-std/
        magnitude channels are on very different scales and need it."""
        self.windows = windows
        self.extra_features = extra_features
        self.norm_mean = norm_mean
        self.norm_std = norm_std

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w = self.windows[idx]
        raw_imu = np.concatenate([w.accel, w.gyro], axis=1).astype(np.float32)  # (T, 6), real physical units
        imu = raw_imu
        if self.extra_features:
            extra = engineer_features(w.accel, w.gyro, w.dt)  # (T, 6)
            imu = np.concatenate([imu, extra], axis=1)  # (T, 12)
        if self.norm_mean is not None:
            imu = (imu - self.norm_mean) / self.norm_std
        # "imu" (possibly extended + z-score normalized) feeds the network;
        # "imu_raw" (always real accel m/s^2 / gyro rad/s) is what
        # train.py's forward_pass integrates through the physics baseline —
        # normalizing that too would integrate z-scored numbers instead of
        # real acceleration, breaking dead-reckoning entirely, not just
        # feature scale. When extra_features/norm are both off, imu_raw and
        # imu are identical (same values as before this was added).
        return {
            "imu": torch.from_numpy(imu),
            "imu_raw": torch.from_numpy(raw_imu),
            "speed_gt": torch.from_numpy(w.speed_gt.astype(np.float32)),
            "pos_gt": torch.from_numpy(w.pos_gt.astype(np.float32)),
            "v0": torch.tensor(w.v0, dtype=torch.float32),
            "theta0": torch.tensor(w.theta0, dtype=torch.float32),
            "lat0": w.lat0,
            "lon0": w.lon0,
            "dt": w.dt,
        }


def compute_channel_norm_stats(windows: list[Window], extra_features: bool) -> tuple[np.ndarray, np.ndarray]:
    """Per-channel mean/std over every timestep of every window, meant to be
    called on the *train* split only and reused (not recomputed) for
    val/test/etc — computing it per-split would leak each split's own
    distribution into its own normalization and isn't how the network would
    see data at real inference time anyway (no access to future/other
    windows' statistics)."""
    all_imu = []
    for w in windows:
        imu = np.concatenate([w.accel, w.gyro], axis=1).astype(np.float32)
        if extra_features:
            imu = np.concatenate([imu, engineer_features(w.accel, w.gyro, w.dt)], axis=1)
        all_imu.append(imu)
    stacked = np.concatenate(all_imu, axis=0)  # (n_windows * T, C)
    mean = stacked.mean(axis=0)
    std = stacked.std(axis=0)
    std[std < 1e-6] = 1.0  # avoid divide-by-zero on a degenerate channel
    return mean.astype(np.float32), std.astype(np.float32)


def load_dataset_splits(data_root: str, variant: str, column_map: dict, sample_rate_hz: float,
                         window_size: int, window_stride: int,
                         train_split: float, val_split: float, seed: int = 0,
                         file_prefix: str = "", extra_features: bool = False):
    """Discover CSV files, load+calibrate+resample each sequence, window
    them, then split at the *file* level into train/val/test.

    file_prefix filters by filename prefix — set to "S-" (see config) to
    train only on smartphone recordings. IO-VNBD mixes vehicle CAN-bus
    files (`V-*.csv`, no 3-axis IMU) into the same folders, and those will
    never resolve accel/gyro columns no matter what — filtering them out
    up front avoids a wall of noisy [skip] lines for files that were never
    going to work, instead of relying on the per-file error to catch it.
    """
    root = Path(data_root) / variant
    csv_paths = sorted(glob.glob(str(root / "**" / f"{file_prefix}*.csv"), recursive=True))
    if not csv_paths:
        raise FileNotFoundError(f"No CSVs found under {root} matching prefix '{file_prefix}' — check data_root/variant/file_prefix in config.")

    rng = np.random.default_rng(seed)
    paths = np.array(csv_paths)
    rng.shuffle(paths)
    n = len(paths)
    n_train = int(n * train_split)
    n_val = int(n * val_split)
    split_paths = {
        "train": paths[:n_train],
        "val": paths[n_train:n_train + n_val],
        "test": paths[n_train + n_val:],
    }

    dt = 1.0 / sample_rate_hz
    split_windows: dict[str, list[Window]] = {}
    for split, files in split_paths.items():
        windows: list[Window] = []
        for p in files:
            try:
                seq = load_sequence(Path(p), column_map)
                seq = resample_uniform(seq, sample_rate_hz)
                seq = recover_yaw_axis(seq)   # IO-VNBD's gyro "Yaw" column is not yaw; see its docstring
                seq = calibrate_sequence(seq)
                windows.extend(build_windows(seq, window_size, window_stride, dt))
            except Exception as e:
                print(f"[skip] {p}: {e}")
        split_windows[split] = windows
        print(f"{split}: {len(files)} files -> {len(windows)} windows")

    # Normalization stats computed from train only (see
    # compute_channel_norm_stats's docstring) — only when extra_features is
    # on, so the existing plain-6-channel path's numbers stay bit-for-bit
    # unchanged from before this was added.
    norm_mean, norm_std = (None, None)
    if extra_features:
        norm_mean, norm_std = compute_channel_norm_stats(split_windows["train"], extra_features=True)

    out = {}
    for split, windows in split_windows.items():
        out[split] = IOVNBDWindowDataset(windows, extra_features=extra_features,
                                          norm_mean=norm_mean, norm_std=norm_std)
    return out


def load_combined_dataset_splits(comma2k19_dir: str | None = None, own_recordings_dir: str | None = None,
                                  decimeter_dir: str | None = None, ppc_dir: str | None = None,
                                  pvs_dir: str | None = None, seed: int = 0, **iovnbd_kwargs):
    """load_dataset_splits(**iovnbd_kwargs), optionally with comma2k19
    windows mixed into the same train/val/test splits — split at the
    *segment* level (same leakage-avoidance reasoning as IO-VNBD's
    file-level split above), so no comma2k19 segment's windows cross a
    split boundary either.

    pvs_dir (e.g. "data/PVS/PVS-Dataset", see src/data/pvs_loader.py)
    mixes in real mixed-speed Brazilian road driving the same additive,
    segment-level-split way ppc_dir does — another real dataset aimed at
    IO-VNBD's urban/low-speed weakness, not a highway-flavored addition.

    ppc_dir (e.g. "data/PPC/PPC-Dataset", see src/data/ppc_loader.py)
    mixes in real urban-Japan driving the same additive, segment-level-
    split way comma2k19_dir does — added specifically to help IO-VNBD's
    urban/low-speed drift number (see diagnose_drift.py), not a
    highway-flavored addition like comma2k19/decimeter.

    comma2k19_dir=None (or no parquet files found there) behaves exactly
    like load_dataset_splits — this is an additive, opt-in extension, not
    a replacement.

    Returns the combined splits dict, plus "iovnbd_test_only" and
    "comma2k19_test_only" — the same two test sets kept separate — so a
    caller can check whether mixing comma2k19 into training measurably
    helped/hurt *IO-VNBD* test drift specifically, not just report a
    combined number that could hide either direction.

    decimeter_dir (e.g. "DECIMETER/sdc2023/sdc2023/train", see
    src/data/decimeter_loader.py) mixes in the Google Smartphone Decimeter
    Challenge dataset the same additive way comma2k19_dir does, with one
    difference: split at the *drive* level (a drive_id groups 1-2 phones
    that rode together and recorded almost the same GPS track), not the
    segment level — splitting two phones from the same drive into different
    train/val/test buckets would leak near-identical trajectories across
    the split, the same category of mistake the file-/segment-level splits
    above already exist to avoid.

    own_recordings_dir (e.g. "data/own_recordings", see DevRecorder /
    data/own_recordings/README.md) mixes real phone recordings in too, but
    deliberately *only into train*, never val/test — unlike comma2k19
    above. These are ad hoc supplementary clips (often just one or two
    files), not a benchmark: splitting a handful of files at the file
    level the way comma2k19 does would, with n this small, plausibly send
    100% of the windows to test and 0% to train — the opposite of the
    point of collecting them. The project's real eval numbers keep coming
    from the untouched IO-VNBD/comma2k19 test sets either way.
    """
    combined = load_dataset_splits(seed=seed, **iovnbd_kwargs)
    out = combined
    # Always available (not just when comma2k19_dir mixes something in) —
    # it's just the untouched IO-VNBD test split, unaffected by whatever
    # else gets mixed into out["test"] below. Needed so any mix (decimeter
    # alone, comma2k19 alone, or both) can still report an isolated
    # IO-VNBD-only number for a fair before/after comparison.
    out["iovnbd_test_only"] = combined["test"]

    sample_rate_hz = iovnbd_kwargs["sample_rate_hz"]
    window_size = iovnbd_kwargs["window_size"]
    window_stride = iovnbd_kwargs["window_stride"]
    dt = 1.0 / sample_rate_hz

    if comma2k19_dir:
        comma_paths = sorted(glob.glob(f"{comma2k19_dir}/*.parquet"))
        if not comma_paths:
            print(f"[combined dataset] no comma2k19 parquet files under {comma2k19_dir} — skipping, IO-VNBD only")
        else:
            from .comma2k19_loader import load_all_segments  # local import: extra deps (pyarrow, huggingface_hub)

            train_split = iovnbd_kwargs["train_split"]
            val_split = iovnbd_kwargs["val_split"]

            seqs = load_all_segments(comma_paths, target_hz=sample_rate_hz)
            rng = np.random.default_rng(seed)
            idxs = np.arange(len(seqs))
            rng.shuffle(idxs)
            n = len(idxs)
            n_train = int(n * train_split)
            n_val = int(n * val_split)
            split_idxs = {"train": idxs[:n_train], "val": idxs[n_train:n_train + n_val], "test": idxs[n_train + n_val:]}

            comma_windows_by_split: dict[str, list[Window]] = {}
            for split, ii in split_idxs.items():
                windows: list[Window] = []
                for i in ii:
                    seq = seqs[i]
                    try:
                        calibrate_sequence(seq)
                        windows.extend(build_windows(seq, window_size, window_stride, dt))
                    except Exception as e:
                        print(f"[skip] comma2k19 segment {seq.path}: {e}")
                comma_windows_by_split[split] = windows
                print(f"comma2k19 {split}: {len(ii)} segments -> {len(windows)} windows")

            # Reuse whatever extra_features/norm stats load_dataset_splits already
            # set up on the IO-VNBD side (computed from IO-VNBD *train* windows only)
            # so comma2k19 windows get the identical treatment — a second,
            # independently-fit normalization would make the two datasets' channels
            # not directly comparable to the network.
            extra_features = combined["train"].extra_features
            norm_mean, norm_std = combined["train"].norm_mean, combined["train"].norm_std

            out = {}
            for split in ("train", "val", "test"):
                out[split] = IOVNBDWindowDataset(list(combined[split].windows) + comma_windows_by_split[split],
                                                  extra_features=extra_features, norm_mean=norm_mean, norm_std=norm_std)
            out["iovnbd_test_only"] = combined["test"]
            out["comma2k19_test_only"] = IOVNBDWindowDataset(comma_windows_by_split["test"],
                                                              extra_features=extra_features, norm_mean=norm_mean, norm_std=norm_std)

    if decimeter_dir:
        dec_root = Path(decimeter_dir)
        if not dec_root.exists():
            print(f"[combined dataset] decimeter_dir {decimeter_dir} does not exist — skipping")
        else:
            from .decimeter_loader import load_all_segments as load_decimeter_segments

            train_split = iovnbd_kwargs["train_split"]
            val_split = iovnbd_kwargs["val_split"]

            dec_seqs = load_decimeter_segments(str(dec_root), target_hz=sample_rate_hz)
            if not dec_seqs:
                print(f"[combined dataset] no usable decimeter segments under {decimeter_dir} — skipping")
            else:
                # Split at the *drive* level (path.parent.name — see this
                # function's docstring) rather than shuffling segments
                # directly, so two phones riding the same drive never end up
                # split across train/val/test.
                drive_ids = np.array(sorted({seq.path.parent.name for seq in dec_seqs}))
                rng2 = np.random.default_rng(seed)
                rng2.shuffle(drive_ids)
                n_drives = len(drive_ids)
                n_train_d = int(n_drives * train_split)
                n_val_d = int(n_drives * val_split)
                drive_split = {
                    "train": set(drive_ids[:n_train_d]),
                    "val": set(drive_ids[n_train_d:n_train_d + n_val_d]),
                    "test": set(drive_ids[n_train_d + n_val_d:]),
                }

                dec_windows_by_split: dict[str, list[Window]] = {"train": [], "val": [], "test": []}
                dec_segments_by_split = {"train": 0, "val": 0, "test": 0}
                for seq in dec_seqs:
                    split = next((s for s, ids in drive_split.items() if seq.path.parent.name in ids), None)
                    if split is None:
                        continue
                    try:
                        calibrate_sequence(seq)
                        dec_windows_by_split[split].extend(build_windows(seq, window_size, window_stride, dt))
                        dec_segments_by_split[split] += 1
                    except Exception as e:
                        print(f"[skip] decimeter segment {seq.path}: {e}")
                for split, windows in dec_windows_by_split.items():
                    print(f"decimeter {split}: {dec_segments_by_split[split]} segments "
                          f"({len(drive_split[split])} drives) -> {len(windows)} windows")

                # Reuse whatever extra_features/norm stats are already set up
                # (IO-VNBD train stats, possibly with comma2k19 already mixed
                # in above) — same reasoning as the comma2k19 block: a second,
                # independently-fit normalization would make datasets'
                # channels not directly comparable to the network.
                extra_features = out["train"].extra_features
                norm_mean, norm_std = out["train"].norm_mean, out["train"].norm_std

                for split in ("train", "val", "test"):
                    out[split] = IOVNBDWindowDataset(list(out[split].windows) + dec_windows_by_split[split],
                                                      extra_features=extra_features, norm_mean=norm_mean, norm_std=norm_std)
                out["decimeter_test_only"] = IOVNBDWindowDataset(dec_windows_by_split["test"],
                                                                   extra_features=extra_features, norm_mean=norm_mean, norm_std=norm_std)

    if ppc_dir:
        ppc_root = Path(ppc_dir)
        if not ppc_root.exists():
            print(f"[combined dataset] ppc_dir {ppc_dir} does not exist — skipping")
        else:
            from .ppc_loader import load_all_segments as load_ppc_segments

            train_split = iovnbd_kwargs["train_split"]
            val_split = iovnbd_kwargs["val_split"]

            ppc_seqs = load_ppc_segments(str(ppc_root), target_hz=sample_rate_hz)
            if not ppc_seqs:
                print(f"[combined dataset] no usable PPC runs under {ppc_dir} — skipping")
            else:
                # Segment-level split (each run is its own independent session,
                # not multiple phones sharing one drive like decimeter) — same
                # reasoning/shape as the comma2k19 block above. Only 6 segments
                # total, so split counts will be small (plausibly 0 in one
                # bucket) — accepted, same as comma2k19 would behave at this N.
                rng3 = np.random.default_rng(seed)
                idxs = np.arange(len(ppc_seqs))
                rng3.shuffle(idxs)
                n = len(idxs)
                n_train = int(n * train_split)
                n_val = int(n * val_split)
                split_idxs = {"train": idxs[:n_train], "val": idxs[n_train:n_train + n_val], "test": idxs[n_train + n_val:]}

                ppc_windows_by_split: dict[str, list[Window]] = {}
                for split, ii in split_idxs.items():
                    windows: list[Window] = []
                    for i in ii:
                        seq = ppc_seqs[i]
                        try:
                            calibrate_sequence(seq)
                            windows.extend(build_windows(seq, window_size, window_stride, dt))
                        except Exception as e:
                            print(f"[skip] PPC run {seq.path}: {e}")
                    ppc_windows_by_split[split] = windows
                    print(f"ppc {split}: {len(ii)} runs -> {len(windows)} windows")

                extra_features = out["train"].extra_features
                norm_mean, norm_std = out["train"].norm_mean, out["train"].norm_std
                for split in ("train", "val", "test"):
                    out[split] = IOVNBDWindowDataset(list(out[split].windows) + ppc_windows_by_split[split],
                                                      extra_features=extra_features, norm_mean=norm_mean, norm_std=norm_std)
                out["ppc_test_only"] = IOVNBDWindowDataset(ppc_windows_by_split["test"],
                                                             extra_features=extra_features, norm_mean=norm_mean, norm_std=norm_std)

    if pvs_dir:
        pvs_root = Path(pvs_dir)
        if not pvs_root.exists():
            print(f"[combined dataset] pvs_dir {pvs_dir} does not exist — skipping")
        else:
            from .pvs_loader import load_all_segments as load_pvs_segments

            train_split = iovnbd_kwargs["train_split"]
            val_split = iovnbd_kwargs["val_split"]

            pvs_seqs = load_pvs_segments(str(pvs_root), target_hz=sample_rate_hz)
            if not pvs_seqs:
                print(f"[combined dataset] no usable PVS folders under {pvs_dir} — skipping")
            else:
                # Segment-level split — same shape as comma2k19/ppc above
                # (each "PVS <n>" folder is its own independent drive, no
                # multi-phone-per-drive grouping concern like decimeter).
                rng4 = np.random.default_rng(seed)
                idxs = np.arange(len(pvs_seqs))
                rng4.shuffle(idxs)
                n = len(idxs)
                n_train = int(n * train_split)
                n_val = int(n * val_split)
                split_idxs = {"train": idxs[:n_train], "val": idxs[n_train:n_train + n_val], "test": idxs[n_train + n_val:]}

                pvs_windows_by_split: dict[str, list[Window]] = {}
                for split, ii in split_idxs.items():
                    windows: list[Window] = []
                    for i in ii:
                        seq = pvs_seqs[i]
                        try:
                            calibrate_sequence(seq)
                            windows.extend(build_windows(seq, window_size, window_stride, dt))
                        except Exception as e:
                            print(f"[skip] PVS folder {seq.path}: {e}")
                    pvs_windows_by_split[split] = windows
                    print(f"pvs {split}: {len(ii)} folders -> {len(windows)} windows")

                extra_features = out["train"].extra_features
                norm_mean, norm_std = out["train"].norm_mean, out["train"].norm_std
                for split in ("train", "val", "test"):
                    out[split] = IOVNBDWindowDataset(list(out[split].windows) + pvs_windows_by_split[split],
                                                      extra_features=extra_features, norm_mean=norm_mean, norm_std=norm_std)
                out["pvs_test_only"] = IOVNBDWindowDataset(pvs_windows_by_split["test"],
                                                             extra_features=extra_features, norm_mean=norm_mean, norm_std=norm_std)

    if own_recordings_dir:
        own_paths = sorted(glob.glob(f"{own_recordings_dir}/*.csv"))
        if not own_paths:
            print(f"[combined dataset] no own_recordings CSVs under {own_recordings_dir} — skipping")
        else:
            own_windows: list[Window] = []
            for p in own_paths:
                try:
                    seq = load_sequence(Path(p))
                    seq = resample_uniform(seq, sample_rate_hz)
                    calibrate_sequence(seq)
                    own_windows.extend(build_windows(seq, window_size, window_stride, dt))
                except Exception as e:
                    print(f"[skip] own_recording {p}: {e}")
            print(f"own_recordings: {len(own_paths)} files -> {len(own_windows)} windows (added to train only)")

            extra_features = out["train"].extra_features
            norm_mean, norm_std = out["train"].norm_mean, out["train"].norm_std
            out["train"] = IOVNBDWindowDataset(list(out["train"].windows) + own_windows,
                                                extra_features=extra_features, norm_mean=norm_mean, norm_std=norm_std)

    return out


In [ ]:
%%writefile src/chain_eval.py
"""
Chained-blackout evaluation: the metric the live app actually lives or dies by.

Per-window drift (train.py / evaluate.py) scores each 5 s window on its own,
starting from the TRUE heading and speed, and only looks at the final
*position*. The fusion engine (fusion.py, FusionEngine.kt) instead chains
5 s chunks, each one starting from the previous chunk's own final
heading/speed. A model can look great per window and still fail that way:
the cleanlabels checkpoint's yaw correction was ~0.03 rad/s over the first
10 samples of a window but ~2 rad/s over the last 10 (nothing in the loss
looked at final heading), so a 30 s chained blackout drifted ~90-100% while
its per-window median was 11%. This module reproduces the chained regime
directly from the held-out windows, so training can select on it.

Chains are rebuilt from back-to-back windows of the same recording
(Window.src / Window.start): chunk j is the window starting exactly
window_size*j samples after chunk 0. Chunk 0 starts from the true state;
chunks 1.. start from the model's own previous final speed/heading/position,
exactly as run_fusion does. Drift = final position error / true distance
travelled over the whole chain, the same definition as the PS metric.
"""
from __future__ import annotations

import numpy as np
import torch


def build_chains(dataset, window_size: int, n_chunks: int, chain_stride_chunks: int = 2,
                 max_chains: int | None = 800, seed: int = 0,
                 speed_range: tuple[float, float] | None = None) -> list[list[int]]:
    """Dataset indices of n_chunks back-to-back windows, per recording.
    chain_stride_chunks spaces chain starts so chains from one recording
    overlap less. Chains with any missing window (dropped as parked / no GT)
    are skipped, never bridged. speed_range=(lo, hi) keeps only chains whose
    mean ground-truth speed (m/s) is in [lo, hi): how a regime-specific model
    is scored only on the kind of driving it was trained for."""
    by_src: dict[str, dict[int, int]] = {}
    for idx, w in enumerate(dataset.windows):
        if w.start >= 0:
            by_src.setdefault(w.src, {})[w.start] = idx
    chains = []
    for starts in by_src.values():
        for st in sorted(starts):
            if (st // window_size) % chain_stride_chunks:
                continue
            ids = [starts.get(st + j * window_size) for j in range(n_chunks)]
            if all(i is not None for i in ids):
                if speed_range is not None:
                    mean_v = float(np.mean([dataset.windows[i].speed_gt.mean() for i in ids]))
                    if not (speed_range[0] <= mean_v < speed_range[1]):
                        continue
                chains.append(ids)
    if max_chains is not None and len(chains) > max_chains:
        rng = np.random.default_rng(seed)
        chains = [chains[i] for i in sorted(rng.choice(len(chains), max_chains, replace=False))]
    return chains


@torch.no_grad()
def chained_drift(model, dataset, device, dt: float, window_size: int = 50, n_chunks: int = 6,
                  max_chains: int | None = 800, v_clamp_max: float = 50.0, batch_size: int = 128,
                  speed_range: tuple[float, float] | None = None,
                  model_hi=None, switch_mps: float = 6.0) -> dict:
    """Chained n_chunks*window_size/10 s blackout drift over the dataset.
    Returns {} if no complete chain exists.

    model_hi: optional second ("duo") model. Each chunk is run by `model`
    (the urban/low-speed model) when the chain's CURRENT estimated speed is
    below switch_mps, else by model_hi (the highway model) — decided from the
    model's own state at the start of the chunk, exactly as fusion.py does."""
    chains = build_chains(dataset, window_size, n_chunks, max_chains=max_chains, speed_range=speed_range)
    if not chains:
        return {}
    model.eval()
    if model_hi is not None:
        model_hi.eval()
    drifts = []
    for b0 in range(0, len(chains), batch_size):
        cb = chains[b0:b0 + batch_size]
        B = len(cb)
        items = [[dataset[i] for i in ids] for ids in cb]
        v = torch.tensor([it[0]["v0"] for it in items], dtype=torch.float32, device=device)
        th = torch.tensor([it[0]["theta0"] for it in items], dtype=torch.float32, device=device)
        p = torch.zeros(B, 2, device=device)
        gt_end = torch.zeros(B, 2, device=device)
        dist = torch.zeros(B, device=device)
        for j in range(n_chunks):
            imu = torch.stack([it[j]["imu"] for it in items]).to(device)
            raw = torch.stack([it[j]["imu_raw"] for it in items]).to(device)
            pos_gt = torch.stack([it[j]["pos_gt"] for it in items]).to(device)
            c = model(imu)
            if model_hi is not None:
                use_hi = (v >= switch_mps).view(-1, 1, 1)
                c = torch.where(use_hi, model_hi(imu), c)
            acc = raw[..., 0] + c[..., 0]
            yaw = raw[..., 5] + c[..., 1]
            # per-step clamp like run_fusion (a window-level cumsum can't clamp mid-way)
            vs = []
            vv = v
            for k in range(acc.shape[1]):
                vv = (vv + acc[:, k] * dt).clamp(0.0, v_clamp_max)
                vs.append(vv)
            speed = torch.stack(vs, dim=1)
            heading = th.unsqueeze(1) + torch.cumsum(yaw * dt, dim=1)
            step = torch.stack([speed * torch.cos(heading), speed * torch.sin(heading)], dim=-1) * dt
            p = p + step.sum(dim=1)
            v, th = speed[:, -1], heading[:, -1]
            gt_end = gt_end + pos_gt[:, -1, :]
            dist = dist + torch.linalg.norm(pos_gt[:, 1:] - pos_gt[:, :-1], dim=-1).sum(dim=1)
        drifts.extend((100.0 * torch.linalg.norm(p - gt_end, dim=-1) / dist.clamp(min=1e-6)).cpu().tolist())
    d = np.array(drifts)
    return {"n_chains": int(len(d)), "chain_seconds": n_chunks * window_size * dt,
            "mean_drift_pct": float(d.mean()), "median_drift_pct": float(np.median(d)),
            "p90_drift_pct": float(np.percentile(d, 90)), "pass_rate_pct": float((d < 10.0).mean() * 100)}


class ChainTrainDataset(torch.utils.data.Dataset):
    """K back-to-back windows of one recording per item, for chained-chunk
    training (train.py --chain_train_chunks). Chunk 0 starts from the true
    state; the training loop carries the model's own end state into chunk 1..
    (see train.chain_loss), so the network practises recovering from its own
    heading/speed errors instead of always being handed the truth."""

    def __init__(self, dataset, chains: list[list[int]]):
        self.dataset = dataset
        self.chains = chains

    def __len__(self):
        return len(self.chains)

    def __getitem__(self, i):
        items = [self.dataset[j] for j in self.chains[i]]
        out = {k: torch.stack([it[k] for it in items]) for k in ("imu", "imu_raw", "speed_gt", "pos_gt")}
        out["v0"], out["theta0"] = items[0]["v0"], items[0]["theta0"]
        return out


In [ ]:
%%writefile src/train.py
"""
Train BiasCorrectionNet end-to-end against real trajectory drift.

Pipeline per window:
    1. network reads calibrated IMU window -> [delta_v, delta_theta] residuals
    2. physics baseline: forward_accel = accel[:,0], yaw_rate = gyro[:,2]
    3. corrected rates = physics + residuals
    4. integrate corrected rates (strapdown_ins.py) -> predicted speed,
       heading, and 2D trajectory for the window
    5. loss = per-step speed error + end-to-end position drift vs GT

Run:
    python -m src.train --config configs/default.yaml
"""
from __future__ import annotations

import argparse
import json
import shutil
import time
from pathlib import Path

import torch
import torch.nn as nn
import yaml
from torch.utils.data import DataLoader

from src.chain_eval import ChainTrainDataset, build_chains, chained_drift
from src.data.windowing import load_combined_dataset_splits
from src.models.bias_correction_net import BiasCorrectionNet
from src.models.strapdown_ins import dead_reckon_position, drift_metric, integrate_heading, integrate_speed


def pick_device(requested: str) -> torch.device:
    if requested != "auto":
        return torch.device(requested)
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def forward_pass(model, batch, dt: float, device):
    imu = batch["imu"].to(device)               # (B, T, 6 or 12) — network input, possibly
                                                 # extended/normalized (see IOVNBDWindowDataset)
    imu_raw = batch["imu_raw"].to(device)        # (B, T, 6) — always real accel/gyro units,
                                                 # used for the physics integration below
    speed_gt = batch["speed_gt"].to(device)     # (B, T)
    pos_gt = batch["pos_gt"].to(device)         # (B, T, 2)
    v0 = batch["v0"].to(device)                 # (B,) true starting speed
    theta0 = batch["theta0"].to(device)         # (B,) true starting heading

    corrections = model(imu)                    # (B, T, 2) -> [delta_v, delta_theta]
    delta_v, delta_theta = corrections[..., 0], corrections[..., 1]

    forward_accel = imu_raw[..., 0] + delta_v    # residual added to raw forward accel
    yaw_rate = imu_raw[..., 5] + delta_theta      # residual added to raw yaw rate (gz)

    # Start from the window's real initial state, not zero — a window is a
    # random slice mid-drive, so the vehicle is essentially never stopped
    # and facing the arbitrary "heading=0" reference right at the slice
    # boundary. Integrating from zero here was the actual bug behind the
    # drift plateau; see windowing.py's Window docstring for the full story.
    speed_pred = integrate_speed(forward_accel, dt, v0=v0)
    heading_pred = integrate_heading(yaw_rate, dt, theta0=theta0)
    pos_pred = dead_reckon_position(speed_pred, heading_pred, dt)

    return speed_pred, pos_pred, speed_gt, pos_gt


def _heading_from_pos(pos: torch.Tensor) -> torch.Tensor:
    """Direction of travel (rad, 0=+x/east — same convention as
    integrate_heading) implied by each consecutive step of a position
    trajectory. Works on both pos_pred and pos_gt: for pos_pred specifically,
    this recovers heading_pred exactly wherever speed_pred > 0 (dead_reckon_
    position's own vx=speed*cos(heading)/vy=speed*sin(heading) construction
    guarantees atan2(vy,vx)=heading), so no separate heading_pred plumbing
    through forward_pass/its 5 other callers is needed."""
    d = pos[:, 1:, :] - pos[:, :-1, :]
    return torch.atan2(d[..., 1], d[..., 0])


def compute_loss(speed_pred, pos_pred, speed_gt, pos_gt, weights: dict):
    speed_loss = nn.functional.mse_loss(speed_pred, speed_gt)
    # drift loss: normalize final position error by distance travelled so it
    # matches the PS's own metric shape (%, not raw metres) and doesn't get
    # swamped by long high-speed windows.
    drift_pct = drift_metric(pos_pred, pos_gt)
    drift_loss = drift_pct.mean()
    metrics = {"speed_loss": speed_loss.item(), "drift_pct": drift_pct.mean().item()}
    total = weights["speed"] * speed_loss + weights["drift"] * drift_loss

    # Real gap found and fixed this session: configs/default.yaml has
    # declared loss_weights.heading=0.5 since before this file's current
    # form, but nothing ever read it — drift_loss (end-to-end position
    # error) was the only signal ever telling the network its *heading*
    # predictions specifically were wrong, an indirect, compounding-prone
    # route to what should be a direct one. Ground truth here is the real
    # direction of travel between consecutive true GPS positions (pos_gt
    # always comes from real lat/lon — see windowing.py — unlike heading_gt,
    # which many IO-VNBD files don't have at all), masked to steps with
    # enough real movement to trust: a near-zero step's implied direction is
    # dominated by GPS noise, not signal, the same low-speed hazard
    # fusion.py's docstring already flags and diagnose_label_noise.py
    # measured directly this session (heading disagreement ~4x worse below
    # 3 m/s than at 6-10 m/s) — supervising heading off a noisy label in
    # exactly the regime that needs it most would fight the fix, not help it.
    heading_weight = weights.get("heading", 0.0)
    if heading_weight > 0:
        min_step_m = weights.get("heading_min_step_m", 0.15)
        step_dist = torch.linalg.norm(pos_gt[:, 1:, :] - pos_gt[:, :-1, :], dim=-1)
        valid = step_dist >= min_step_m
        if valid.any():
            heading_pred = _heading_from_pos(pos_pred)
            heading_gt = _heading_from_pos(pos_gt)
            diff = heading_pred - heading_gt
            circular_err = torch.atan2(torch.sin(diff), torch.cos(diff))  # wrap to (-pi, pi]
            heading_loss = (circular_err[valid] ** 2).mean()
        else:
            heading_loss = torch.zeros((), device=pos_pred.device, dtype=pos_pred.dtype)
        total = total + heading_weight * heading_loss
        metrics["heading_loss"] = heading_loss.item()

    # Final-heading term. The per-step term above covers every step, but the
    # failure that matters for the live app is the heading a window ENDS on:
    # fusion.py / FusionEngine.kt start each 5 s blackout chunk from the
    # previous chunk's final heading. The cleanlabels checkpoint (heading
    # loss off) put ~2 rad/s of yaw "correction" into the last 10 samples of
    # a window: the per-window drift number barely noticed (it only reads the
    # final POSITION), but chained over 30 s the estimate spun in circles
    # (~90-100% drift on all 64 comma2k19 segments vs 17% median for the old
    # checkpoint). Direction of travel over the last `heading_end_steps`
    # steps, predicted vs real, wrapped to (-pi, pi]; masked to windows that
    # really moved over that span (same noise argument as above).
    end_weight = weights.get("heading_end", 0.0)
    if end_weight > 0:
        k = int(weights.get("heading_end_steps", 10))
        d_pred = pos_pred[:, -1, :] - pos_pred[:, -1 - k, :]
        d_gt = pos_gt[:, -1, :] - pos_gt[:, -1 - k, :]
        valid = torch.linalg.norm(d_gt, dim=-1) >= weights.get("heading_end_min_m", 1.0)
        if valid.any():
            diff = torch.atan2(d_pred[..., 1], d_pred[..., 0]) - torch.atan2(d_gt[..., 1], d_gt[..., 0])
            end_loss = (torch.atan2(torch.sin(diff), torch.cos(diff))[valid] ** 2).mean()
        else:
            end_loss = torch.zeros((), device=pos_pred.device, dtype=pos_pred.dtype)
        total = total + end_weight * end_loss
        metrics["heading_end_loss"] = end_loss.item()

    return total, metrics


def chain_loss(model, batch, dt: float, device, weights: dict):
    """Chained-chunk training loss over K back-to-back windows.

    The live fusion engine starts every 5 s blackout chunk from the PREVIOUS
    chunk's own final speed/heading/position, but single-window training
    always starts from the truth, so nothing ever taught the network to
    recover from its own end-state error (the cleanlabels yaw blow-up and the
    ~35-43% chained drift of chainfix both come from that). Here chunk 0
    starts from the true state, chunk j>0 from chunk j-1's predicted
    end state WITH gradients, so an early heading error is punished by every
    later chunk it throws off. Per chunk: the usual compute_loss (speed +
    in-chunk drift + heading terms), plus `chain` x the cumulative drift %
    (error of the summed chunk displacements vs the summed true ones over the
    distance travelled so far, the same definition chain_eval.chained_drift
    and the PS metric use), averaged over chunks."""
    imu = batch["imu"].to(device)            # (B, K, T, C)
    raw = batch["imu_raw"].to(device)
    speed_gt = batch["speed_gt"].to(device)  # (B, K, T)
    pos_gt = batch["pos_gt"].to(device)      # (B, K, T, 2)
    v = batch["v0"].to(device)
    th = batch["theta0"].to(device)
    B, K = imu.shape[0], imu.shape[1]
    p_cum = torch.zeros(B, 2, device=device)
    gt_cum = torch.zeros(B, 2, device=device)
    dist = torch.zeros(B, device=device)
    chain_w = weights.get("chain", 1.0)
    total, metrics = 0.0, {}
    last_chain_drift = None
    for j in range(K):
        c = model(imu[:, j])
        acc = raw[:, j, :, 0] + c[..., 0]
        yaw = raw[:, j, :, 5] + c[..., 1]
        speed = integrate_speed(acc, dt, v0=v)
        heading = integrate_heading(yaw, dt, theta0=th)
        pos = dead_reckon_position(speed, heading, dt)
        loss_j, m = compute_loss(speed, pos, speed_gt[:, j], pos_gt[:, j], weights)
        p_cum = p_cum + pos[:, -1]
        gt_cum = gt_cum + pos_gt[:, j, -1]
        dist = dist + torch.linalg.norm(pos_gt[:, j, 1:] - pos_gt[:, j, :-1], dim=-1).sum(dim=1)
        cd = 100.0 * torch.linalg.norm(p_cum - gt_cum, dim=-1) / dist.clamp(min=1e-6)
        total = total + loss_j + chain_w * cd.mean()
        for k, val in m.items():
            metrics[k] = metrics.get(k, 0.0) + val / K
        v, th = speed[:, -1], heading[:, -1]
        last_chain_drift = cd
    metrics["chain_drift_pct"] = last_chain_drift.mean().item()   # cumulative over all K chunks
    return total / K, metrics


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", default="configs/default.yaml")
    parser.add_argument(
        "--resume", default=None,
        help="Warm-start from an existing checkpoint (e.g. checkpoints/best.pt) instead "
             "of random init. The loaded weights' val drift is measured once up front so "
             "best-checkpoint tracking / early stopping stay honest about whether this run "
             "actually beats what it started from. The previous best.pt and train_history.json "
             "are backed up (*_prev) before anything gets overwritten.",
    )
    parser.add_argument(
        "--comma2k19_dir", default=None,
        help="Mix comma2k19 windows (see src/data/comma2k19_loader.py) into train/val/test "
             "alongside IO-VNBD, e.g. --comma2k19_dir data/comma2k19_demo/data. Opt-in — "
             "omitted (the default) trains on IO-VNBD only, unchanged from before.",
    )
    parser.add_argument(
        "--decimeter_dir", default=None,
        help="Mix Google Smartphone Decimeter Challenge windows (see "
             "src/data/decimeter_loader.py) into train/val/test alongside "
             "IO-VNBD (and comma2k19, if given), e.g. --decimeter_dir "
             "DECIMETER/sdc2023/sdc2023/train. Opt-in — omitted (the "
             "default) leaves training unchanged.",
    )
    parser.add_argument(
        "--ppc_dir", default=None,
        help="Mix PPC-Dataset windows (see src/data/ppc_loader.py) into "
             "train/val/test alongside IO-VNBD (and comma2k19/decimeter, if "
             "given), e.g. --ppc_dir data/PPC/PPC-Dataset. Real urban-Japan "
             "driving, added specifically to help IO-VNBD's urban/low-speed "
             "drift number (see diagnose_drift.py) — not highway-flavored "
             "like comma2k19/decimeter. Opt-in — omitted (the default) "
             "leaves training unchanged.",
    )
    parser.add_argument(
        "--pvs_dir", default=None,
        help="Mix PVS-Dataset windows (see src/data/pvs_loader.py) into "
             "train/val/test alongside IO-VNBD (and whatever else is given), "
             "e.g. --pvs_dir data/PVS/PVS-Dataset. Real mixed-speed Brazilian "
             "road driving, added for the same reason as --ppc_dir — helping "
             "the urban/low-speed number, not highway-flavored. Opt-in — "
             "omitted (the default) leaves training unchanged.",
    )
    parser.add_argument(
        "--own_recordings_dir", default=None,
        help="Mix real phone recordings (see DevRecorder / "
             "data/own_recordings/README.md) into training, e.g. "
             "--own_recordings_dir data/own_recordings. Added to *train only* "
             "(never val/test) — these are ad hoc supplementary clips, not a "
             "benchmark, unlike --comma2k19_dir. Opt-in — omitted (the "
             "default) trains on IO-VNBD (+ comma2k19 if given) only, "
             "unchanged from before.",
    )
    parser.add_argument(
        "--extra_features", action="store_true",
        help="Append windowing.py's engineer_features() 6 extra channels (accel/gyro "
             "magnitude, jerk, local smoothing+roughness) to the raw 6, z-score-normalized "
             "per-channel using train-split statistics (see IOVNBDWindowDataset). First "
             "attempt at this (no normalization, low first-cycle patience) was inconclusive "
             "— see engineer_features()'s docstring. Automatically bumps "
             "model.input_channels to 12 regardless of the config value.",
    )
    parser.add_argument(
        "--heading_weight", type=float, default=None,
        help="Override configs/default.yaml's loss_weights.heading (0.5) for this run — "
             "see compute_loss's doc for what this term does. 0.5 turned out, empirically, "
             "to hurt both the full model and the urban-regime model on a real run (both "
             "landed worse than their own no-heading-loss baselines) — it was never tuned "
             "before this session since the weight was previously declared but dead code "
             "(see compute_loss's doc), so 0.5 has no empirical basis. Pass a smaller value "
             "(e.g. 0.05) to test a lighter touch, or 0.0 to disable it entirely without "
             "editing the config. Omitted (the default) uses whatever the config says.",
    )
    parser.add_argument(
        "--heading_end_weight", type=float, default=None,
        help="Override configs/default.yaml's loss_weights.heading_end for this run (the "
             "final-heading term in compute_loss; 0 disables it).",
    )
    parser.add_argument(
        "--chain_train_chunks", type=int, default=0,
        help="Train on K back-to-back 5 s windows per sample (see chain_loss) instead of one, "
             "carrying the model's own end state from chunk to chunk. 0 = off (the old "
             "single-window objective). 3 is the setting used for the chaintrain run.",
    )
    parser.add_argument(
        "--chain_train_max", type=int, default=40000,
        help="Cap on training chains per epoch when --chain_train_chunks is on (seeded random subset).",
    )
    parser.add_argument(
        "--chain_chunks", type=int, default=6,
        help="Chained-blackout validation length in 5 s chunks (6 = a 30 s blackout, the "
             "regime fusion.py/the app actually runs; see src/chain_eval.py). Checkpoint "
             "selection uses val window drift + this chained median drift. 0 disables it "
             "and selects on window drift only, as before.",
    )
    parser.add_argument(
        "--run_name", default=None,
        help="Save/load under checkpoints/best_<run_name>.pt and "
             "results/train_history_<run_name>.json instead of the plain best.pt / "
             "train_history.json — use for any experimental run (different "
             "input_channels, architecture, etc.) so it can't silently overwrite the "
             "real best.pt with a checkpoint of an incompatible shape. Omitted (the "
             "default) behaves exactly as before.",
    )
    parser.add_argument(
        "--speed_regime", choices=["urban", "highway"], default=None,
        help="Train a regime-specific model instead of one shared model across both — "
             "see diagnose_drift.py's confirmed finding that mixing highway-flavored data "
             "(comma2k19, decimeter) measurably hurts IO-VNBD's low-speed drift number, "
             "evidence of a real shared-capacity tradeoff in one small model, not a bug. "
             "'urban' keeps train/val windows with mean speed_gt below --urban_max_speed_mps; "
             "'highway' keeps windows at or above it. The test split is left untouched (not "
             "filtered) so this checkpoint's eval_report stays directly comparable against "
             "the existing full-test numbers already in docs/understanding.md. Omitted (the "
             "default) trains on every window, unchanged from before.",
    )
    parser.add_argument(
        "--urban_max_speed_mps", type=float, default=6.0,
        help="Threshold for --speed_regime, in m/s (a window's OWN mean speed_gt, not a "
             "per-sample cutoff). Only read when --speed_regime is set.",
    )
    parser.add_argument(
        "--seed", type=int, default=42,
        help="Real gap found and fixed this session: nothing here ever seeded torch's own "
             "RNG (model weight init, dropout, DataLoader shuffling) — only the data *split* "
             "was seeded (load_combined_dataset_splits' own separate seed=0). Every run, even "
             "with identical data/config, took a different stochastic path, so before/after "
             "comparisons across runs (this session included several) carry an unknown amount "
             "of pure random-init noise on top of whatever real effect was being measured. "
             "Fixed value now (was previously unset/nondeterministic) so two runs with the "
             "same flags are actually comparable. Change only if you deliberately want a "
             "different draw (e.g. to sanity-check how much a result varies by seed alone).",
    )
    args = parser.parse_args()
    torch.manual_seed(args.seed)
    if args.extra_features:
        print("--extra_features: using 12-channel input (6 raw + 6 engineered), "
              "z-score normalized from train-split stats")

    cfg = yaml.safe_load(open(args.config))
    if args.heading_weight is not None:
        cfg["train"]["loss_weights"]["heading"] = args.heading_weight
        print(f"--heading_weight: overriding loss_weights.heading -> {args.heading_weight}")
    if args.heading_end_weight is not None:
        cfg["train"]["loss_weights"]["heading_end"] = args.heading_end_weight
        print(f"--heading_end_weight: overriding loss_weights.heading_end -> {args.heading_end_weight}")
    device = pick_device(cfg["train"]["device"])
    print(f"device: {device}")

    splits = load_combined_dataset_splits(
        comma2k19_dir=args.comma2k19_dir,
        decimeter_dir=args.decimeter_dir,
        ppc_dir=args.ppc_dir,
        pvs_dir=args.pvs_dir,
        own_recordings_dir=args.own_recordings_dir,
        data_root=cfg["data"]["root"],
        variant=cfg["data"]["variant"],
        column_map=cfg["data"]["column_map"],
        sample_rate_hz=cfg["data"]["sample_rate_hz"],
        window_size=cfg["data"]["window_size"],
        window_stride=cfg["data"]["window_stride"],
        train_split=cfg["data"]["train_split"],
        val_split=cfg["data"]["val_split"],
        file_prefix=cfg["data"].get("file_prefix", ""),
        extra_features=args.extra_features,
    )

    # Chained validation must score the FULL val set (a regime filter keeps
    # only some windows, which fragments chains), restricted to chains whose
    # own mean speed is in this model's regime.
    import copy
    val_full = copy.copy(splits["val"])
    chain_speed_range = None
    if args.speed_regime == "urban":
        chain_speed_range = (0.0, args.urban_max_speed_mps)
    elif args.speed_regime == "highway":
        chain_speed_range = (args.urban_max_speed_mps, float("inf"))

    if args.speed_regime:
        def keep(w):
            mean_speed = float(w.speed_gt.mean())
            return (mean_speed < args.urban_max_speed_mps) if args.speed_regime == "urban" \
                else (mean_speed >= args.urban_max_speed_mps)

        # train/val only — test stays the full, untouched split (see the
        # arg's own doc) so this checkpoint's eval_report is directly
        # comparable against every other checkpoint's numbers, not scored
        # against an easier/harder subset of its own choosing.
        for split_name in ("train", "val"):
            before = len(splits[split_name].windows)
            splits[split_name].windows = [w for w in splits[split_name].windows if keep(w)]
            after = len(splits[split_name].windows)
            print(f"[speed_regime={args.speed_regime}] {split_name}: {before} -> {after} windows "
                  f"(threshold {args.urban_max_speed_mps} m/s)")

    train_loader = DataLoader(splits["train"], batch_size=cfg["train"]["batch_size"], shuffle=True)
    if args.chain_train_chunks > 0:
        train_chains = build_chains(splits["train"], cfg["data"]["window_size"], args.chain_train_chunks,
                                    chain_stride_chunks=2, max_chains=args.chain_train_max, seed=args.seed)
        print(f"chained-chunk training: {len(train_chains)} chains x {args.chain_train_chunks} chunks "
              f"({args.chain_train_chunks * 5} s each)")
        train_loader = DataLoader(ChainTrainDataset(splits["train"], train_chains),
                                  batch_size=max(8, cfg["train"]["batch_size"] // args.chain_train_chunks), shuffle=True)
    val_loader = DataLoader(splits["val"], batch_size=cfg["train"]["batch_size"], shuffle=False)

    input_channels = 12 if args.extra_features else cfg["model"]["input_channels"]
    model = BiasCorrectionNet(
        input_channels=input_channels,
        cnn_channels=cfg["model"]["cnn_channels"],
        cnn_kernel_size=cfg["model"]["cnn_kernel_size"],
        gru_hidden=cfg["model"]["gru_hidden"],
        gru_layers=cfg["model"]["gru_layers"],
        dropout=cfg["model"]["dropout"],
        output_dim=cfg["model"]["output_dim"],
    ).to(device)

    ckpt_dir = Path("checkpoints")
    ckpt_dir.mkdir(exist_ok=True)
    results_dir = Path("results")
    results_dir.mkdir(exist_ok=True)
    suffix = f"_{args.run_name}" if args.run_name else ""
    best_name = f"best{suffix}.pt"
    history_path = results_dir / f"train_history{suffix}.json"
    dt = 1.0 / cfg["data"]["sample_rate_hz"]

    def chain_metrics(m):
        if args.chain_chunks <= 0:
            return {}
        m.eval()
        r = chained_drift(m, val_full, device, dt, window_size=cfg["data"]["window_size"],
                          n_chunks=args.chain_chunks, speed_range=chain_speed_range)
        return {"chain_median": r["median_drift_pct"], "chain_mean": r["mean_drift_pct"],
                "chain_pass": r["pass_rate_pct"], "chain_n": r["n_chains"]} if r else {}

    def selection_score(window_drift, chain):
        # window drift alone is blind to chained heading failures (see
        # chain_eval.py); add the chained median so a checkpoint can't win
        # on one while failing the other.
        return window_drift + chain.get("chain_median", 0.0)

    epoch_offset = 0
    if args.resume:
        print(f"resuming from {args.resume}")
        try:
            model.load_state_dict(torch.load(args.resume, map_location=device))
        except (RuntimeError, OSError) as e:
            # RuntimeError: most likely the checkpoint was trained with a
            # different model.input_channels (an architecture/feature
            # change) and its layer shapes no longer match. OSError: path
            # doesn't exist / unreadable. Either way, an unattended loop
            # (train_until_target.sh, the Colab loop cell) shouldn't crash
            # over it — fall back to a cold start instead.
            print(f"could not load {args.resume} ({e}); "
                  f"starting from random init instead (likely an input_channels/architecture change or missing checkpoint).")
            args.resume = None  # so the loop below knows this run is effectively a cold start

    if args.resume:
        # Back up whatever the previous run left behind before this run
        # overwrites best.pt / train_history.json — resuming should never
        # silently destroy the checkpoint/history it started from.
        best_path = ckpt_dir / best_name
        if best_path.exists():
            shutil.copyfile(best_path, ckpt_dir / f"best{suffix}_prev.pt")
        prev_history = []
        if history_path.exists():
            prev_history = json.load(open(history_path))
            shutil.copyfile(history_path, results_dir / f"train_history{suffix}_prev.json")
            epoch_offset = (prev_history[-1]["epoch"] + 1) if prev_history else 0

        # Measure the loaded weights' actual val drift before training so
        # "best" tracking below is honest about whether this run improves on
        # what it started from, instead of resetting to inf and overwriting
        # best.pt with something worse the moment val drift dips even once.
        model.eval()
        # Not hardcoded to {"speed_loss", "drift_pct"} — compute_loss can
        # return extra keys (e.g. "heading_loss", only when loss_weights.
        # heading > 0, see its own doc) that a fixed-key dict would KeyError
        # on the first accumulation. .get(k, 0.0) makes this robust to
        # whatever metrics compute_loss actually returns.
        baseline: dict[str, float] = {}
        with torch.no_grad():
            for batch in val_loader:
                speed_pred, pos_pred, speed_gt, pos_gt = forward_pass(model, batch, dt, device)
                _, metrics = compute_loss(speed_pred, pos_pred, speed_gt, pos_gt, cfg["train"]["loss_weights"])
                for k, v in metrics.items():
                    baseline[k] = baseline.get(k, 0.0) + v
        for k in baseline:
            baseline[k] /= max(1, len(val_loader))
        base_chain = chain_metrics(model)
        baseline.update(base_chain)
        print(f"resumed weights baseline val drift: {baseline['drift_pct']:.2f}%"
              + (f" | chained {args.chain_chunks * 5}s median {base_chain['chain_median']:.2f}% "
                 f"(n={base_chain['chain_n']})" if base_chain else ""))
    else:
        prev_history = []
        baseline = None

    opt = torch.optim.AdamW(model.parameters(), lr=cfg["train"]["lr"], weight_decay=cfg["train"]["weight_decay"])
    # Flat LR the whole run was a real gap — a run showed steadily shrinking
    # per-epoch improvement (-1.4% -> -0.6% -> -0.4% -> -0.2%...) consistent
    # with the fixed step size overshooting near a minimum rather than the
    # model having genuinely stopped learning. Cosine decay lets it keep
    # taking finer steps as training progresses instead of asking one LR to
    # work well for both the beginning and the end of the run. On a resume,
    # this restarts the cosine cycle from the configured peak LR (a "warm
    # restart") rather than continuing the decayed tail of the previous run
    # — deliberately, since a fully-decayed LR has nowhere left to explore.
    #
    # Bug found after 7 identical-result warm-restart cycles in a row
    # (train_until_target_comma.sh, 2026-09-05): T_max was always
    # cfg["train"]["epochs"] (60), but early_stop_patience=8 was cutting
    # every *resumed* cycle off after only 8-16 real epochs — nowhere near
    # 60 — so LR barely moved off its 1.0e-3 peak (e.g. 1.00e-03 -> 9.67e-04
    # over 8 epochs) before the cycle ended. Every restart was therefore
    # retracing almost the same high-LR trajectory for a similar short
    # duration and landing in the same place — that's *why* the cycles kept
    # matching, not evidence the model was maxed out.
    #
    # First attempt used T_max = patience*3 (24), reasoning it'd decay
    # nicely *if* a cycle ran long. It didn't help: a resumed cycle that
    # never finds a new best always stops at exactly `patience` epochs
    # (no improvement to reset the counter), so the guaranteed worst-case
    # window is `early_stop_patience` epochs, not something a longer T_max
    # can lean on. Sized to the guaranteed window instead, so LR reaches a
    # genuinely low, fine-tuning-scale value even in that worst case
    # (T_max=10, patience=8 -> LR ~2e-4 by the 8th epoch instead of ~9.7e-4).
    # Cold start keeps the full-length cycle (unchanged; that run already
    # reached the 63.26% best cleanly over 57 real epochs, finding enough
    # new bests along the way to actually use a long cycle).
    cycle_len = cfg["train"]["epochs"] if not args.resume else max(cfg["train"]["early_stop_patience"] + 2, 10)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cycle_len)

    # Seed "best" from the loaded checkpoint's real val drift (not inf) so
    # this run only overwrites best.pt when it actually beats what it
    # started from.
    best_val_drift = baseline["drift_pct"] if baseline is not None else float("inf")
    best_score = selection_score(baseline["drift_pct"], baseline) if baseline is not None else float("inf")
    patience = cfg["train"]["early_stop_patience"]
    bad_epochs = 0
    history = []

    for i in range(cfg["train"]["epochs"]):
        epoch = epoch_offset + i
        model.train()
        t0 = time.time()
        # Not hardcoded — see baseline's identical doc above (compute_loss
        # can return extra keys like "heading_loss" that a fixed-key dict
        # would KeyError on).
        train_metrics: dict[str, float] = {}
        for batch in train_loader:
            opt.zero_grad()
            if args.chain_train_chunks > 0:
                loss, metrics = chain_loss(model, batch, dt, device, cfg["train"]["loss_weights"])
            else:
                speed_pred, pos_pred, speed_gt, pos_gt = forward_pass(model, batch, dt, device)
                loss, metrics = compute_loss(speed_pred, pos_pred, speed_gt, pos_gt, cfg["train"]["loss_weights"])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            for k, v in metrics.items():
                train_metrics[k] = train_metrics.get(k, 0.0) + v
        for k in train_metrics:
            train_metrics[k] /= max(1, len(train_loader))

        model.eval()
        val_metrics: dict[str, float] = {}
        with torch.no_grad():
            for batch in val_loader:
                speed_pred, pos_pred, speed_gt, pos_gt = forward_pass(model, batch, dt, device)
                _, metrics = compute_loss(speed_pred, pos_pred, speed_gt, pos_gt, cfg["train"]["loss_weights"])
                for k, v in metrics.items():
                    val_metrics[k] = val_metrics.get(k, 0.0) + v
        for k in val_metrics:
            val_metrics[k] /= max(1, len(val_loader))

        val_chain = chain_metrics(model)
        val_metrics.update(val_chain)
        score = selection_score(val_metrics["drift_pct"], val_chain)
        dt_epoch = time.time() - t0
        extra = ""
        if "heading_end_loss" in val_metrics:
            extra += f" | end-heading loss {val_metrics['heading_end_loss']:.3f}"
        if val_chain:
            extra += f" | chained {args.chain_chunks * 5}s median {val_chain['chain_median']:.2f}% pass {val_chain['chain_pass']:.0f}%"
        print(f"epoch {epoch:03d} | train drift {train_metrics['drift_pct']:.2f}% "
              f"| val drift {val_metrics['drift_pct']:.2f}%{extra} | score {score:.2f} "
              f"| lr {scheduler.get_last_lr()[0]:.2e} | {dt_epoch:.1f}s")
        history.append({"epoch": epoch, "train": train_metrics, "val": val_metrics, "score": score})
        json.dump(prev_history + history, open(history_path, "w"), indent=2)
        scheduler.step()

        if score < best_score:
            best_score = score
            best_val_drift = val_metrics["drift_pct"]
            bad_epochs = 0
            torch.save(model.state_dict(), ckpt_dir / best_name)
            print(f"  -> new best score {best_score:.2f} (val drift {best_val_drift:.2f}%), saved checkpoints/{best_name}")
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"early stopping at epoch {epoch} (no score improvement for {patience} epochs)")
                break

    print(f"best val drift this run: {best_val_drift:.2f}% -> checkpoints/{best_name} "
          f"(previous best backed up at checkpoints/best{suffix}_prev.pt)" if args.resume else
          f"best val drift: {best_val_drift:.2f}% -> checkpoints/{best_name}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/evaluate.py
"""
Evaluate a trained model against the actual SIH26168 benchmark criteria
(doc §2 / §5):
    - positional drift < 10% of distance travelled during GNSS blackout
    - example checks: <5m drift over 50m in <1min; <100m drift over 1km @60kmph

Reports both the aggregate drift-% (same metric trained on) and the two
named example scenarios, on the held-out test split, sequence by sequence
so you can see which trip types the model struggles on (tight turns are
usually the failure mode for heading-rate errors) before the demo.

Run:
    python -m src.evaluate --config configs/default.yaml --checkpoint checkpoints/best.pt
    # with comma2k19 mixed in (matches how it was trained, if --comma2k19_dir was used):
    python -m src.evaluate --config configs/default.yaml --checkpoint checkpoints/best.pt --comma2k19_dir data/comma2k19_demo/data
    # with decimeter mixed in (matches how it was trained, if --decimeter_dir was used):
    python -m src.evaluate --config configs/default.yaml --checkpoint checkpoints/best_decimeter.pt --decimeter_dir DECIMETER/sdc2023/sdc2023/train --run_name decimeter
"""
from __future__ import annotations

import argparse
import json

import numpy as np
import torch
import yaml
from torch.utils.data import DataLoader

from src.chain_eval import chained_drift
from src.data.windowing import load_combined_dataset_splits
from src.models.bias_correction_net import BiasCorrectionNet
from src.train import forward_pass, pick_device
from src.models.strapdown_ins import drift_metric


def eval_drift(model, dataset, dt, device, batch_size, target_pct=None):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_drift = []
    with torch.no_grad():
        for batch in loader:
            speed_pred, pos_pred, speed_gt, pos_gt = forward_pass(model, batch, dt, device)
            all_drift.extend(drift_metric(pos_pred, pos_gt).cpu().numpy().tolist())
    all_drift = np.array(all_drift)
    report = {
        "n_windows": len(all_drift),
        "mean_drift_pct": float(all_drift.mean()),
        "median_drift_pct": float(np.median(all_drift)),
        "p90_drift_pct": float(np.percentile(all_drift, 90)),
        "worst_drift_pct": float(all_drift.max()) if len(all_drift) else float("nan"),
    }
    if target_pct is not None:
        report["target_pct"] = target_pct
        report["pass_rate_pct"] = float((all_drift < target_pct).mean() * 100)
    return report


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", default="configs/default.yaml")
    parser.add_argument("--checkpoint", default="checkpoints/best.pt")
    parser.add_argument(
        "--comma2k19_dir", default=None,
        help="Also break out IO-VNBD-only vs comma2k19-only test drift separately, not just "
             "the combined number — pass the same value used for src.train's --comma2k19_dir.",
    )
    parser.add_argument(
        "--decimeter_dir", default=None,
        help="Also break out decimeter-only test drift separately — pass the same value "
             "used for src.train's --decimeter_dir, e.g. DECIMETER/sdc2023/sdc2023/train.",
    )
    parser.add_argument(
        "--ppc_dir", default=None,
        help="Also break out PPC-Dataset-only test drift separately — pass the same value "
             "used for src.train's --ppc_dir, e.g. data/PPC/PPC-Dataset.",
    )
    parser.add_argument(
        "--pvs_dir", default=None,
        help="Also break out PVS-Dataset-only test drift separately — pass the same value "
             "used for src.train's --pvs_dir, e.g. data/PVS/PVS-Dataset.",
    )
    parser.add_argument(
        "--extra_features", action="store_true",
        help="Must match whatever the checkpoint was trained with — see src.train's "
             "--extra_features. Rebuilds the same 12-channel normalized input.",
    )
    parser.add_argument(
        "--run_name", default=None,
        help="Save reports as results/eval_report_<run_name>.json / "
             "eval_report_breakdown_<run_name>.json instead of the plain names. Evaluating "
             "any checkpoint other than checkpoints/best.pt without this silently "
             "overwrites the production checkpoint's own saved report with whatever "
             "checkpoint you just happened to pass — found this happen for real "
             "(eval_report.json briefly held the engineered-features run's numbers "
             "instead of best.pt's). Omitted (the default) behaves exactly as before.",
    )
    args = parser.parse_args()

    cfg = yaml.safe_load(open(args.config))
    device = pick_device(cfg["train"]["device"])

    splits = load_combined_dataset_splits(
        comma2k19_dir=args.comma2k19_dir,
        decimeter_dir=args.decimeter_dir,
        ppc_dir=args.ppc_dir,
        pvs_dir=args.pvs_dir,
        data_root=cfg["data"]["root"],
        variant=cfg["data"]["variant"],
        column_map=cfg["data"]["column_map"],
        sample_rate_hz=cfg["data"]["sample_rate_hz"],
        window_size=cfg["data"]["window_size"],
        window_stride=cfg["data"]["window_stride"],
        train_split=cfg["data"]["train_split"],
        val_split=cfg["data"]["val_split"],
        file_prefix=cfg["data"].get("file_prefix", ""),
        extra_features=args.extra_features,
    )

    input_channels = 12 if args.extra_features else cfg["model"]["input_channels"]
    model = BiasCorrectionNet(
        input_channels=input_channels,
        cnn_channels=cfg["model"]["cnn_channels"],
        cnn_kernel_size=cfg["model"]["cnn_kernel_size"],
        gru_hidden=cfg["model"]["gru_hidden"],
        gru_layers=cfg["model"]["gru_layers"],
        dropout=cfg["model"]["dropout"],
        output_dim=cfg["model"]["output_dim"],
    ).to(device)
    model.load_state_dict(torch.load(args.checkpoint, map_location=device))
    model.eval()

    dt = 1.0 / cfg["data"]["sample_rate_hz"]
    target = cfg["eval"]["drift_target_pct"]
    batch_size = cfg["train"]["batch_size"]
    suffix = f"_{args.run_name}" if args.run_name else ""

    report = eval_drift(model, splits["test"], dt, device, batch_size, target_pct=target)

    # Chained 30 s blackout (what the live app does) next to the per-window
    # number — see src/chain_eval.py for why the per-window one alone is not
    # enough. Test split, same checkpoint.
    chain_report = {}
    window_size = cfg["data"]["window_size"]
    chain_all = chained_drift(model, splits["test"], device, dt, window_size=window_size, n_chunks=6, max_chains=None)
    if chain_all:
        report["chained_30s"] = chain_all
        chain_report["all"] = chain_all

    print(json.dumps(report, indent=2))
    json.dump(report, open(f"results/eval_report{suffix}.json", "w"), indent=2)

    if report["mean_drift_pct"] < target:
        print(f"\n✅ mean drift {report['mean_drift_pct']:.2f}% is under the {target}% PS target.")
    else:
        print(f"\n⚠️  mean drift {report['mean_drift_pct']:.2f}% is OVER the {target}% PS target — "
              f"needs more training/tuning before demo day.")

    # iovnbd_test_only is always present once anything else is mixed in (see
    # load_combined_dataset_splits); comma2k19_test_only / decimeter_test_only
    # only show up if the matching --*_dir was actually passed to this run.
    # Checked independently (not all-or-nothing) so e.g. a decimeter-only
    # mix still gets its own honest per-dataset breakdown without needing
    # comma2k19 in the same run.
    per_dataset = {}
    for key in ("iovnbd_test_only", "comma2k19_test_only", "decimeter_test_only", "ppc_test_only", "pvs_test_only"):
        if key in splits:
            per_dataset[key] = eval_drift(model, splits[key], dt, device, batch_size, target_pct=target)
            ch = chained_drift(model, splits[key], device, dt, window_size=window_size, n_chunks=6, max_chains=None)
            if ch:
                per_dataset[key]["chained_30s"] = ch
    if per_dataset:
        print("\n--- breakdown by dataset (same checkpoint) ---")
        print(json.dumps(per_dataset, indent=2))
        json.dump(per_dataset, open(f"results/eval_report_breakdown{suffix}.json", "w"), indent=2)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/export_onnx.py
"""
Export the trained BiasCorrectionNet to ONNX for on-device inference.

Why export only the network, not the physics integrator: strapdown_ins.py's
integration is trivial arithmetic (a cumulative sum) that the mobile app's
own real-time loop should do sample-by-sample as new IMU data arrives —
that's also how "instant seamless mode-switching" between GNSS and pure-INS
works in practice (you can't wait for a whole window to re-integrate).
So the exported artifact's contract with the mobile app is:

    ONNX model:  (1, window_size, 6) calibrated IMU window -> (1, window_size, 2) [delta_v, delta_theta]
    App-side:    forward_accel + delta_v[-1], yaw_rate + delta_theta[-1]  ->  integrate one step

Dynamic axes are left off the batch dim on purpose — ONNX Runtime Mobile
does one window at a time on-device, so a fixed batch size of 1 keeps the
exported graph simpler and avoids surprises in the mobile runtime.

Run:
    python -m src.export_onnx --config configs/default.yaml --checkpoint checkpoints/best.pt
"""
from __future__ import annotations

import argparse

import onnx
import onnxruntime as ort
import torch
import yaml

from src.models.bias_correction_net import BiasCorrectionNet


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", default="configs/default.yaml")
    parser.add_argument("--checkpoint", default="checkpoints/best.pt")
    args = parser.parse_args()

    cfg = yaml.safe_load(open(args.config))
    model = BiasCorrectionNet(
        input_channels=cfg["model"]["input_channels"],
        cnn_channels=cfg["model"]["cnn_channels"],
        cnn_kernel_size=cfg["model"]["cnn_kernel_size"],
        gru_hidden=cfg["model"]["gru_hidden"],
        gru_layers=cfg["model"]["gru_layers"],
        dropout=cfg["model"]["dropout"],
        output_dim=cfg["model"]["output_dim"],
    )
    model.load_state_dict(torch.load(args.checkpoint, map_location="cpu"))
    model.eval()

    window_size = cfg["data"]["window_size"]
    dummy = torch.randn(1, window_size, cfg["model"]["input_channels"])
    out_path = cfg["export"]["onnx_path"]

    torch.onnx.export(
        model,
        dummy,
        out_path,
        input_names=["imu_window"],
        output_names=["corrections"],
        opset_version=cfg["export"]["opset"],
        dynamic_axes=None,  # fixed shape, see module docstring
    )

    # sanity-check: structural validity + numerical parity vs. the torch model
    onnx_model = onnx.load(out_path)
    onnx.checker.check_model(onnx_model)

    sess = ort.InferenceSession(out_path, providers=["CPUExecutionProvider"])
    torch_out = model(dummy).detach().numpy()
    onnx_out = sess.run(None, {"imu_window": dummy.numpy()})[0]
    max_diff = abs(torch_out - onnx_out).max()
    print(f"exported to {out_path}")
    print(f"max |torch - onnx| output diff: {max_diff:.2e} (should be ~1e-5 or smaller)")

    if max_diff > 1e-3:
        raise RuntimeError("ONNX export diverges from the PyTorch model beyond tolerance — do not ship this.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile configs/default.yaml
# SIH26168 — Dead Reckoning ML config
# NOTE: column_map is filled in once we inspect the actual IO-VNBD CSV headers
# (data is git-lfs, pulled separately — see README). Placeholder names below
# are updated by src/data/inspect_dataset.py on first run.

data:
  root: "data/IO-VNBD"
  # which dataset variant to train on first: the synchronised set has vehicle
  # (V) + smartphone (S) IMU streams time-aligned, which we need for the
  # phone-mounted use case described in the PS.
  variant: "Synchronised V abd S datasets"
  # V-*.csv files are vehicle CAN-bus data with no 3-axis accel/gyro (only
  # 2-axis "indicated" accel + yaw rate) — not usable here. S-*.csv are the
  # real smartphone recordings (3-axis accel+gyro+GPS), which is what this
  # PS is actually about. Confirmed against real headers, 2026-09-03.
  file_prefix: "S-"
  sample_rate_hz: 10          # PS explicitly specifies 10Hz phone update rate
  window_size: 50             # 5s windows @ 10Hz
  window_stride: 10           # 1s stride between training windows
  train_split: 0.7
  val_split: 0.15
  test_split: 0.15
  # filled in by inspect_dataset.py — do not hand-edit until confirmed
  column_map:
    time: null
    accel_x: null
    accel_y: null
    accel_z: null
    gyro_x: null
    gyro_y: null
    gyro_z: null
    speed_gt: null      # wheel-speed or GPS-derived speed, ground truth
    lat: null
    lon: null
    heading_gt: null

model:
  # physics-informed learned bias-correction network:
  #   v_physics = integrate(calibrated forward accel)      <- drifts
  #   v_corrected = v_physics + NN(imu_window)              <- learned residual
  # loss combines per-step speed error + end-to-end drift over the window.
  input_channels: 6           # ax, ay, az, gx, gy, gz (calibrated/leveled)
  # A 6-added-engineered-channel variant (magnitude/jerk/local-std — see
  # windowing.py's engineer_features()) was tried and reverted here; see
  # that function's docstring for the inconclusive result and why.
  # A bigger "v2" architecture (dilated residual CNN + deeper GRU) was also
  # tried and reverted — see bias_correction_net.py's module docstring
  # (71.35% test drift, worse than this one's 62.76%).
  cnn_channels: [32, 64, 64]
  cnn_kernel_size: 5
  gru_hidden: 128
  gru_layers: 2
  dropout: 0.2
  output_dim: 2               # [delta_v (speed residual), delta_theta (heading-rate residual)]

train:
  batch_size: 64
  epochs: 60
  lr: 1.0e-3
  weight_decay: 1.0e-5
  loss_weights:
    speed: 1.0
    # Heading terms are ON. History: the per-step term (`heading`) was first
    # wired up at 0.5 and 0.05, looked neutral-to-negative on per-window
    # drift, and was left at 0. That comparison was blind to the failure
    # that matters: with heading loss off, the shipped-candidate checkpoint
    # (cleanlabels) learned ~2 rad/s yaw "corrections" in the last few
    # samples of each window, invisible to a loss that only reads the final
    # position, and spun in circles when chunks were chained (30 s blackout
    # drift ~90-100% on all 64 comma2k19 segments). 0.5 and 0.05 were also
    # tiny next to a loss in % (drift weight 2 x ~15 = 30), so they could
    # barely move anything. Weights below are in rad^2 on the same scale.
    # Checkpoints are now selected on window drift + chained 30 s drift
    # (src/chain_eval.py), so a heading-loss weight that hurts is caught.
    heading: 10.0
    heading_end: 20.0       # final-heading term; see train.compute_loss. Weights are in rad^2 against a % drift loss, so they look large on purpose.
    chain: 1.0              # cumulative chained drift term, only used with --chain_train_chunks (see train.chain_loss)
    drift: 2.0               # end-to-end position-drift term — this is what the SIH benchmark actually grades
  early_stop_patience: 8
  device: "auto"              # auto -> mps (Apple Silicon) > cuda > cpu

eval:
  # SIH26168 benchmark targets (from PS, doc §2):
  drift_target_pct: 10.0      # < 10% of distance travelled during blackout
  example_checks:
    - {distance_m: 50,  max_drift_m: 5,  max_time_s: 60}
    - {distance_m: 1000, max_drift_m: 100, speed_kmph: 60}

export:
  onnx_path: "checkpoints/dead_reckoning_model.onnx"
  opset: 17


## 5. Confirm the real IO-VNBD column names

**Do not skip this** — the loader fuzzy-matches column names as a fallback,
but confirm it actually resolved the right ones before trusting any result.


In [ ]:
import pandas as pd
from pathlib import Path

# S-*.csv only — those are the smartphone recordings this pipeline trains
# on. V-*.csv (vehicle CAN-bus) lack 3-axis accel/gyro and are excluded via
# configs/default.yaml's data.file_prefix.
samples = sorted(Path("data/IO-VNBD").rglob("S-*.csv"))
samples = [p for p in samples if p.stat().st_size > 5000][:5]

for p in samples:
    print("---", p, "---")
    try:
        df = pd.read_csv(p, nrows=3, encoding="utf-8")
    except UnicodeDecodeError:
        df = pd.read_csv(p, nrows=3, encoding="latin-1")  # IO-VNBD isn't consistently UTF-8
    print(list(df.columns))
    print(df.head(2))
    print()


If the fuzzy-matched columns look wrong, edit `configs/default.yaml`'s
`data.column_map` in the cell above (re-run that `%%writefile` cell with the
real names filled in) before training.


## 6. Get the checkpoint to resume from

Same scheme as `scripts/train_until_target.sh`: the loop warm-starts from
`checkpoints/best_v3chain.pt` (the checkpoint the app ships, see
`checkpoints/README.md`) and writes its own results to
`checkpoints/best_loop.pt` / `results/eval_report_loop.json`, so it can never
overwrite `best_v3chain.pt` or the older `best.pt`. This cell gets, in order:
1. `best_loop.pt` from Google Drive, if a previous Colab run left one (Step 2)
2. `best_v3chain.pt` from the GitHub repo (the starting point)
3. if neither is reachable, Step 7 trains from scratch

`src.train`'s `--resume` flag (below) then warm-starts from whatever this
cell finds, instead of random init.


In [ ]:
import os
import shutil
import urllib.request

os.makedirs("checkpoints", exist_ok=True)
os.makedirs("results", exist_ok=True)

GITHUB_RAW = "https://raw.githubusercontent.com/ShadowIzzHerZ/SIH-2k2026/main"
RESUME_FROM = "checkpoints/best_v3chain.pt"  # the shipped model
RUN_NAME = "loop"
OUT = f"checkpoints/best_{RUN_NAME}.pt"     # what the loop trains into
REPORT = f"results/eval_report_{RUN_NAME}.json"

drive_ok = os.path.exists("/content/drive/MyDrive")
if drive_ok and os.path.exists(CKPT_DIR + f"/best_{RUN_NAME}.pt"):
    shutil.copy(CKPT_DIR + f"/best_{RUN_NAME}.pt", OUT)
    if os.path.exists(RESULTS_DIR + f"/train_history_{RUN_NAME}.json"):
        shutil.copy(RESULTS_DIR + f"/train_history_{RUN_NAME}.json", f"results/train_history_{RUN_NAME}.json")
    print(f"resuming from {OUT} on Google Drive (previous Colab run)")
try:
    urllib.request.urlretrieve(GITHUB_RAW + "/" + RESUME_FROM, RESUME_FROM)
    print(f"downloaded {RESUME_FROM} from the GitHub repo")
except Exception as e:
    print(f"could not download {RESUME_FROM} ({e})")
if not os.path.exists(OUT) and not os.path.exists(RESUME_FROM):
    print("no checkpoint found anywhere — Step 7 will train from scratch")


## 7. Train until the target drift is hit (or this runtime ends)

Each cycle runs a full `src.train` pass with `--run_name loop` (resuming from
`best_loop.pt` once one exists, otherwise from `best_v3chain.pt`) — a
cosine-annealed "warm restart" from the current best weights — then
evaluates on the real held-out test set. `src.train` only writes
`best_loop.pt` when it beats the weights it started from. It keeps looping,
checkpointing to Drive after every cycle so nothing is lost if Colab
disconnects mid-run, until either the PS's <10% drift target is hit or the
cycle cap below is reached (a safety backstop against a runaway loop, not a
real target — at ~60 epochs/cycle it's far more cycles than one Colab
session will ever reach, so in practice this only stops on target-hit,
failure, or the runtime itself ending).

`TRAIN_ARGS` is passed through to `src.train`, e.g.
`["--chain_train_chunks", "3"]` to keep the chained-chunk objective
`best_v3chain.pt` was fine-tuned with.


In [ ]:
import subprocess
import sys
import json
import shutil
import os

TARGET_DRIFT_PCT = 10.0
MAX_CYCLES = 500  # safety backstop only, see markdown above — not a real limit
TRAIN_ARGS = []


def current_checkpoint():
    # best_loop.pt once this loop has improved on its starting point, else
    # the starting point itself (None = nothing yet, cold start)
    for p in (OUT, RESUME_FROM):
        if os.path.exists(p):
            return p
    return None


cycle = 0
while cycle < MAX_CYCLES:
    cycle += 1
    start = current_checkpoint()
    resume_args = ["--resume", start] if start else []
    print(f"\n{'='*70}\ncycle {cycle} {('(warm restart from ' + start + ')') if start else '(cold start)'}\n{'='*70}")

    # -u: unbuffered stdout. Without it, src.train's stdout is piped (not a
    # TTY) so Python fully block-buffers it — nothing appears in the Colab
    # cell until the whole cycle's subprocess exits and flushes everything
    # at once, which can look like a silent hang for many minutes on a
    # cold-start cycle. -u makes per-epoch progress genuinely live here too.
    train_ret = subprocess.run([sys.executable, "-u", "-m", "src.train", "--config", "configs/default.yaml",
                                *resume_args, "--run_name", RUN_NAME, *TRAIN_ARGS])
    if train_ret.returncode != 0:
        print("training subprocess exited with an error — stopping the loop, see output above")
        break

    if drive_ok and os.path.exists(OUT):
        shutil.copy(OUT, CKPT_DIR + f"/best_{RUN_NAME}.pt")
        if os.path.exists(f"results/train_history_{RUN_NAME}.json"):
            shutil.copy(f"results/train_history_{RUN_NAME}.json", RESULTS_DIR + f"/train_history_{RUN_NAME}.json")

    ckpt = current_checkpoint()
    eval_ret = subprocess.run(
        [sys.executable, "-m", "src.evaluate", "--config", "configs/default.yaml", "--checkpoint", ckpt,
         "--run_name", RUN_NAME],
        capture_output=True, text=True,
    )
    print(eval_ret.stdout[-1500:])
    if eval_ret.returncode != 0 or not os.path.exists(REPORT):
        print("evaluate failed or produced no report — stopping the loop, see output above")
        break

    report = json.load(open(REPORT))
    mean_drift = report["mean_drift_pct"]
    print(f"cycle {cycle}: {ckpt} test mean drift {mean_drift:.2f}% (target < {TARGET_DRIFT_PCT}%)")
    if drive_ok:
        shutil.copy(REPORT, RESULTS_DIR + f"/eval_report_{RUN_NAME}.json")

    if mean_drift < TARGET_DRIFT_PCT:
        print(f"\n🎯 target reached after {cycle} cycle(s) — stopping.")
        break
else:
    print(f"\nhit the {MAX_CYCLES}-cycle safety cap without reaching target — this would be very unusual; check results/train_history_{RUN_NAME}.json for what's actually happening (plateaued vs. still improving).")

ckpt = current_checkpoint()
assert ckpt, "no checkpoint at all — something is wrong, scroll up"
print(f"\n✅ {ckpt}: {os.path.getsize(ckpt)} bytes"
      + ("" if ckpt == OUT else f" (no cycle beat {RESUME_FROM}, so there is no {OUT})"))


## 8. Evaluate against the PS's <10% drift benchmark

(Step 7's loop already ran this each cycle — this cell just re-confirms the
final number on the checkpoint the loop stopped on: `best_loop.pt`, or
`best_v3chain.pt` if no cycle improved on it.)


In [ ]:
import os
ckpt = OUT if os.path.exists(OUT) else RESUME_FROM
assert os.path.exists(ckpt), f"{ckpt} doesn't exist — go back and get Step 7 to finish with a ✅ first."
!python -m src.evaluate --config configs/default.yaml --checkpoint {ckpt} --run_name {RUN_NAME}


## 9. Export to ONNX (for the mobile app)

Exports `best_loop.pt` to `checkpoints/dead_reckoning_model_loop.onnx`, not
to the config's `checkpoints/dead_reckoning_model.onnx`: that name is the
model the app ships, and copying a Colab export over it by accident would
replace it with whatever this run produced. Compare
`results/eval_report_loop.json` against `results/README.md` first, then copy
it over the shipped file by hand if it really is better. If no cycle beat
`best_v3chain.pt`, there is nothing new to export.


In [ ]:
import os
import shutil
import yaml

ONNX_OUT = f"checkpoints/dead_reckoning_model_{RUN_NAME}.onnx"
if not os.path.exists(OUT):
    print(f"{OUT} doesn't exist (no cycle beat {RESUME_FROM}) — nothing new to export; "
          f"the shipped ONNX is already {RESUME_FROM}.")
else:
    export_cfg = yaml.safe_load(open("configs/default.yaml"))
    export_cfg["export"]["onnx_path"] = ONNX_OUT
    with open(f"configs/export_{RUN_NAME}.yaml", "w") as f:
        yaml.safe_dump(export_cfg, f)
    !python -m src.export_onnx --config configs/export_{RUN_NAME}.yaml --checkpoint {OUT}
    if os.path.exists(ONNX_OUT) and drive_ok:
        shutil.copy(ONNX_OUT, CKPT_DIR + f"/dead_reckoning_model_{RUN_NAME}.onnx")
        print("ONNX model copied to Drive")


## 10. Download artifacts directly (alternative to Drive)

In [ ]:
from google.colab import files
import os

for f in [OUT, f"checkpoints/dead_reckoning_model_{RUN_NAME}.onnx", REPORT]:
    if os.path.exists(f):
        files.download(f)
    else:
        print(f"skipping {f} — doesn't exist (an earlier step didn't finish)")
